In [ ]:
import sys
import os
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, AIMessage, SystemMessage
from langchain_teddynote import logging

from dotenv import load_dotenv

load_dotenv(override=True)

# 추적을 위한 프로젝트 이름 설정
logging.langsmith("Samsung-Asset-AI-Portal")

# 프롬프트

### system prompt

In [ ]:
_SYSTEM_PROMPT = """당신은 자산운용사에서 변액일임펀드 설정/해지 지시서 처리를 담당하는 오퍼레이터 입니다.
    당신의 역할은 수익자가 메일로 보내온 변액일임펀드 설정/해지 지시서를 시스템에 입력하기 전에
    변액일임펀드 설정/해지 지시서에서 확정분과 청구분을 구분하여 확정분에 대한 설정/해지 데이터와 청구분에 대한 설정/해지 데이터를 수집하고 정리하는 역할입니다.
"""

_SYSTEM_PROMPT_ENG = """You are an operator at an asset management company responsible for processing variable managed fund setup/termination instructions.

Your role is to, before entering the instructions into the system, 
distinguish between confirmed items and pending/claim items in the variable managed fund setup/termination instruction sent by the beneficiary via email, 
and collect and organize the setup/termination data for both the confirmed portion and the pending/claim portion.
"""


### PDF text to markdown 프롬프트

In [ ]:
#pdf 텍스트를 markdown 형식으로 변환하는 프롬프트
_CREATE_PDF_TEXT_TO_MARKDOWN_RULE_PROMPT = """
  - docling 추출 data를 기반으로 전체 내용을 분석하세요.

  - 모든 필드와 모든 데이터 및 모든 텍스트를 최대한 빠짐없이 모두 정리하세요.(설정/해지와 상관없는 텍스트도 모두 정리할 것)

  - 한자가 발견되면 반드시 한글로 변환하세요.

  - 통합 또는 요약하지 말고, 종목(펀드)명과 펀드코드 단위로 data를 정리하세요.

  - 모든 메타 데이터, 테이블 컬럼, 필드들의 의미와 기능을 분석하여 정규화 하고 테이블로 정리하세요.

  - 모든 텍스트(종목(펀드)명과 펀드코드와 상관없는 텍스트 포함)들도 의미를 분석하여 정규화 하고 테이블로 정리하세요.

  - 각 단어와 코드, 숫자 데이터들을 pdfplumber 추출 data와 비교하여 보다 정확한 data를 선택하세요.

  - PDF에서 추출한 data의 특성상 인접한 컬럼의 데이터가 중복되거나, 인접한 컬럼으로 병합되는 오류가 발생할 수 있습니다. 정리 결과의 테이블에서 데이터가 인접한 컬럼에 중복되거나 병합되어 작성되어 있는지 pdfplumber 추출 data와 비교하여 확인하세요. 중복 또는 병합되어 있으면 수정하세요.

  - PDF에서 추출한 data의 특성상 마지막 row의 데이터가 상위 row의 데이터와 병합되는 오류가 발생할 수 있습니다. 테이블의 흐름을 분석하여 정리 결과의 테이블에서 마지막 row의 데이터가 상위 row의 데이터와 병합되어 작성되어 있는지 pdfplumber 추출 data와 비교하여 확인하세요. 병합되어 있으면 수정하세요.

  - PDF에서 추출한 data의 특성상 공백, 띄어쓰기, 줄바꿈 오류가 발생할 수 있습니다. 단어의 의미를 분석하고 맥락을 통해 공백, 띄어쓰기, 줄바꿈 오류가 존재하는지 확인하고 오류가 있으면 단어의 의미와 맥락에 맞도록 수정하세요.

  - 종목명에서 공백, 띄어쓰기, 줄바꿈 오류가 자주 발생합니다. 종목명에서 공백, 띄어쓰기, 줄바꿈 오류가 존재하는지 확인하고 오류가 있으면 단어의 의미와 맥락에 맞도록 수정하세요.

  - 정리 결과에서 단어 또는 숫자를 임의로 제거하지 말고 있는 그대로 출력하세요.

  - 정리 결과에서 단어 또는 숫자를 임의로 요약하지 말고 있는 그대로 출력하세요.

  - 정리 결과에서 단어 또는 숫자를 임의로 새로 작성하지 말고 있는 그대로 출력하세요.

  - 종목명에서 약어를 사용하는지 판단하여 약어를 유지하세요.

  - 펀드 코드와 펀드명이 정확한지 확인하고 작성하세요.(합계 행과 혼동되지 않도록 주의)

  - 펀드 코드를 기준으로 펀드 개수를 집계하세요.(펀드 코드가 없는 경우 펀드 개수를 집계하지 않음)

  - 설정건이 존재할 경우, 데이터의 맥락과 의미를 분석하여 설정건에 대한 매입통보일자를 반드시 추출하세요.

  - 해지건이 존재할 경우, 데이터의 맥락과 의미를 분석하여 해지건에 대한 환매신청일을 반드시 추출하세요.

  - 문서 제목을 발견하면 최상단에 출력하세요.

  - 원문을 번역하지 말고 원문 그대로 출력하세요.
"""


_VALIDATE_PDF_TEXT_TO_MARKDOWN_RULE_PROMPT = """
  - 한자가 발견되면 반드시 한글로 변환하세요.

  - 모든 필드와 텍스트가 정확히 추출되었는지 확안하세요.

  - 정리 결과에서 누락된 필드와 데이터가 있는지 확인하세요.

  - 펀드코드를 기준으로 펀드 개수와 거래 건수가 정확하게 집계되었는지 확안하세요.

  - 펀드코드를 기준으로 펀드 행과 합계 행이 정확히 구분되어 집계되었는지 확안하세요.

  - 모든 항목에서 종목명(펀드명)과 펀드코드가 정확하게 작성되었는지 확안하세요.

  - 정리 결과에서 금액과 합계 데이터가 있는지 확인하세요. 합계가 정확한지 검증하세요.

  - 설정건이 존재할 경우, 설정건에 대한 매입통보일자가 추출되었는지 반드시 확안하세요.

  - 해지건이 존재할 경우, 해지건에 대한 환매신청일자가 추출되었는지 반드시 확안하세요.

  - PDF에서 추출한 data의 특성상 인접한 컬럼의 데이터가 중복되거나, 인접한 컬럼으로 병합되는 오류가 발생할 수 있습니다. 정리 결과의 테이블에서 데이터가 인접한 컬럼에 중복되거나 병합되어 작성되어 있는지 pdfplumber 추출 data와 비교하여 확인하세요.

  - PDF에서 추출한 data의 특성상 마지막 row의 데이터가 상위 row의 데이터와 병합되는 오류가 발생할 수 있습니다. 테이블의 흐름을 분석하여 정리 결과의 테이블에서 마지막 row의 데이터가 상위 row의 데이터와 병합되어 작성되어 있는지 pdfplumber 추출 data와 비교하여 확인하세요.

  - PDF에서 추출한 data의 특성상 공백, 띄어쓰기, 줄바꿈 오류가 발생할 수 있습니다. 단어의 의미를 분석하고 맥락을 통해 공백, 띄어쓰기, 줄바꿈 오류가 존재하는지 확인하세요.

  - 종목명에서 공백, 띄어쓰기, 줄바꿈 오류가 자주 발생합니다. 종목명에서 공백, 띄어쓰기, 줄바꿈 오류가 존재하는지 확인하세요.

  - Markdown 코드의 오류 여부를 검수하여 오류가 발견되면 수정하세요.

  - 검수 결과를 확인하고 오류가 발견되면 오류 항목을 수정하세요.
"""


# pdf 텍스트를 markdown 형식으로 변환하는 프롬프트
def get_prompt_pdf_text_to_markdown(pdf_text_docling: str, pdf_text_pdfplumber: str) -> str:
  prompt_text = f"""
  아래는 수익자가 보내온 변액일임펀드 설정/해지 지시서 PDF 파일에서 pdfplumber와 docling 라이브러리를 사용하여 추출한 data입니다.


  아래의 추출 data를 LLM 모델이 잘 이해할 수 있도록 정리 지침에 따라 정리하세요.

  정리 지침에 따라 정리한 내용을 검수 지침에 따라 검수하세요.

  검수 지침에 따라 검수한 내용을 markdown 형식으로 작성하세요.



  # 변액일임펀드 설정/해지 지시서 PDF 파일 내용 - docling 라이브러리 사용

  {pdf_text_docling}



  # 변액일임펀드 설정/해지 지시서 PDF 파일 내용 - pdfplumber 라이브러리 사용

  {pdf_text_pdfplumber}



  # 반드시 지켜야 할 정리 지침

  {_CREATE_PDF_TEXT_TO_MARKDOWN_RULE_PROMPT}


  # 검수 지침
  {_VALIDATE_PDF_TEXT_TO_MARKDOWN_RULE_PROMPT}

  """
  return prompt_text

  

### excel, text to markdown

In [ ]:
_CREATE_TEXT_TO_MARKDOWN_RULE_PROMPT = """
  - 추출 data를 기반으로 전체 내용을 분석하세요.

  - 모든 필드와 모든 데이터 및 모든 텍스트를 최대한 빠짐없이 모두 정리하세요.(설정/해지와 상관없는 텍스트도 모두 정리할 것)

  - 한자가 발견되면 반드시 한글로 변환하세요.

  - 통합 또는 요약하지 말고, 종목(펀드)명과 펀드코드 단위로 data를 정리하세요.

  - 모든 메타데이터, 테이블 컬럼, 필드들의 의미와 기능을 분석하여 정규화 하고 테이블로 정리하세요.

  - 모든 텍스트(종목(펀드)명과 펀드코드와 상관없는 텍스트 포함)들도 의미를 분석하여 정규화 하고 테이블로 정리하세요.

  - 정리 결과의 테이블에서 데이터가 인접한 컬럼에 중복되거나 병합되어 작성되어 있는지 확인하세요. 중복 또는 병합되어 있으면 수정하세요.

  - 테이블의 흐름을 분석하여 정리 결과의 테이블에서 마지막 row의 데이터가 상위 row의 데이터와 병합되어 작성되어 있는지 확인하세요. 병합되어 있으면 수정하세요.

  - 단어의 의미를 분석하고 맥락을 통해 공백, 띄어쓰기, 줄바꿈 오류가 존재하는지 확인하고 오류가 있으면 단어의 의미와 맥락에 맞도록 수정하세요.

  - 종목명에서 공백, 띄어쓰기, 줄바꿈 오류가 자주 발생합니다. 종목명에서 공백, 띄어쓰기, 줄바꿈 오류가 존재하는지 확인하고 오류가 있으면 단어의 의미와 맥락에 맞도록 수정하세요.

  - 정리 결과에서 단어 또는 숫자를 임의로 제거하지 말고 있는 그대로 출력하세요.

  - 정리 결과에서 단어 또는 숫자를 임의로 요약하지 말고 있는 그대로 출력하세요.

  - 정리 결과에서 단어 또는 숫자를 임의로 새로 작성하지 말고 있는 그대로 출력하세요.

  - 종목명에서 약어를 사용하는지 판단하여 약어를 유지하세요.

  - 펀드 코드와 펀드명이 정확한지 확인하고 작성하세요.(합계 행과 혼동되지 않도록 주의)

  - 펀드 코드를 기준으로 펀드 개수를 집계하세요.(펀드 코드가 없는 경우 펀드 개수를 집계하지 않음)

  - 설정건이 존재할 경우, 데이터의 맥락과 의미를 분석하여 설정건에 대한 매입통보일자를 반드시 추출하세요.

  - 해지건이 존재할 경우, 데이터의 맥락과 의미를 분석하여 해지건에 대한 환매신청일을 반드시 추출하세요.

  - 문서 제목을 발견하면 최상단에 출력하세요.

  - 원문을 번역하지 말고 원문 그대로 출력하세요.
"""

_VALIDATE_TEXT_TO_MARKDOWN_RULE_PROMPT = """
  - 한자가 발견되면 반드시 한글로 변환하세요.

  - 모든 필드와 텍스트가 정확히 추출되었는지 확인한다.

  - 정리 결과에서 누락된 필드와 데이터가 있는지 확인하세요.

  - 펀드코드를 기준으로 펀드 개수와 거래 건수가 정확하게 집계되었는지 확인한다.

  - 펀드코드를 기준으로 펀드 행과 합계 행이 정확히 구분되어 집계되었는지 확인한다.

  - 모든 항목에서 종목명(펀드명)과 펀드코드가 정확하게 작성되었는지 확인한다.

  - 정리 결과에서 금액과 합계 데이터가 있는지 확인하세요. 합계가 정확한지 검증하세요.
  
  - 설정건이 존재할 경우, 설정건에 대한 매입통보일자가 추출되었는지 반드시 확안한다.

  - 해지건이 존재할 경우, 해지건에 대한 환매신청일자가 추출되었는지 반드시 확안한다.

  - 정리 결과의 테이블에서 데이터가 인접한 컬럼에 중복되거나 병합되어 작성되어 있는지 확인하세요.

  - 테이블의 흐름을 분석하여 정리 결과의 테이블에서 마지막 row의 데이터가 상위 row의 데이터와 병합되어 작성되어 있는지 확인하세요.

  - 단어의 의미를 분석하고 맥락을 통해 공백, 띄어쓰기, 줄바꿈 오류가 존재하는지 확인하세요.

  - 종목명에서 공백, 띄어쓰기, 줄바꿈 오류가 자주 발생합니다. 종목명에서 공백, 띄어쓰기, 줄바꿈 오류가 존재하는지 확인하세요.

  - Markdown 코드의 오류 여부를 검수하여 오류가 발견되면 수정한다.

  - 검수 결과를 확인하고 오류가 발견되면 오류 항목을 수정하세요.
"""


def get_prompt_text_to_markdown(original_text: str):
  prompt_text = f"""
  아래는 수익자가 보내온 변액일임펀드 설정/해지 지시서 파일에서 추출한 data입니다.

  아래의 추출 data를 LLM 모델이 잘 이해할 수 있도록 정리 지침에 따라 정리하세요.

  정리 지침에 따라 정리한 내용을 검수 지침에 따라 검수하세요.

  검수 지침에 따라 검수한 내용을 markdown 형식으로 작성하세요.



  ### 변액일임펀드 설정/해지 지시서 파일 내용 ###

  {original_text}



  # 반드시 지켜야 할 중요 지침

  {_CREATE_TEXT_TO_MARKDOWN_RULE_PROMPT}


  # 검수 지침
  {_VALIDATE_PDF_TEXT_TO_MARKDOWN_RULE_PROMPT}

  """
  return prompt_text

### PDF to markdown validation 프롬프트

In [ ]:



# pdf 텍스트를 markdown 형식으로 변환한 데이터가 지침에 따라 올바르게 작성되었는지 검수하는 프롬프트
def get_prompt_pdf_text_to_markdown_validate(text_to_markdown: str, pdf_text_pdfplumber: str, pdf_text_docling: str) -> str:
    prompt_text = f"""
    아래는 수익자가 보내온 변액일임펀드 설정/해지 지시서 PDF 파일에서 text를 추출하여 markdown 형식으로 정리하여 작성한 data입니다.

    원본 PDF 파일에서 추출한 text와 비교하여 markdown 형식으로 정리한 data가 아래의 작성 지침에 따라 올바르게 작성되었는지 검수 지침에 따라 검수하세요.

    검수가 완료되면 검수 결과가 반영된 markdown을 반드시 출력형식에 따라 출력하세요.


    # 변액일임펀드 설정/해지 지시서 markdown 형식 정리 내용

    {text_to_markdown}


    # 변액일임펀드 설정/해지 지시서 PDF 파일 내용 - docling 라이브러리 사용

    {pdf_text_docling}
   

    # 변액일임펀드 설정/해지 지시서 PDF 파일 내용 - pdfplumber 라이브러리 사용

    {pdf_text_pdfplumber}


    # 검수 지침(반드시 수행)

    {_VALIDATE_PDF_TEXT_TO_MARKDOWN_RULE_PROMPT} 


    # 작성 지침

    {_CREATE_PDF_TEXT_TO_MARKDOWN_RULE_PROMPT}


    # 출력 형식 (***반드시 순서대로 markdown 형식으로 출력하고 아래의 내용만 출력할 것***)

      1. 매입통보일 / 환매신청일

      2. 거래 지시 내용 테이블

      3. 전체 거래 집계 현황 테이블

      4. 전체 펀드코드 개수, 거래 개수 집계 현황 테이블

      5. 메타데이터 설명 테이블

      6. 검수 결과 보고서

    # IMPORTANT: 검수 결과 보고서를 확인하고 오류가 있으면 반드시 수정하고 다시 출력하세요!

    """
    return prompt_text

### excel, text to markdown validation 프롬프트

In [ ]:

def get_prompt_text_to_markdown_validate(text_to_markdown: str, original_text: str):
  prompt_text = f"""
    아래는 수익자가 보내온 변액일임펀드 설정/해지 지시서 파일에서 text를 추출하여 markdown 형식으로 정리하여 작성한 data입니다.

    원본 파일에서 추출한 text와 비교하여 markdown 형식으로 정리한 data가 아래의 작성 지침에 따라 올바르게 작성되었는지 검수 지침에 따라 검수하세요.

    검수가 완료되면 검수 결과가 반영된 markdown을 출력하세요.


    # 변액일임펀드 설정/해지 지시서 markdown 형식 정리 내용

    {text_to_markdown}


    # 변액일임펀드 설정/해지 지시서 파일 내용

    {original_text}


    # 검수 지침(반드시 수행)

    {_VALIDATE_TEXT_TO_MARKDOWN_RULE_PROMPT} 


    # 작성 지침

    {_CREATE_TEXT_TO_MARKDOWN_RULE_PROMPT}


    # 출력 형식 (***반드시 순서대로 markdown 형식으로 출력하고 아래의 내용만 출력할 것***)

      1. 매입통보일 / 환매신청일

      2. 거래 지시 내용 테이블

      3. 전체 거래 집계 현황 테이블

      4. 전체 펀드코드 개수, 거래 개수 집계 현황 테이블

      5. 메타데이터 설명 테이블

      6. 검수 결과 보고서

    # IMPORTANT: 검수 결과 보고서를 확인하고 오류가 있으면 반드시 수정하고 다시 출력하세요!
      
  """
  return prompt_text

### 확정분/청구분 구분 지침 프롬프트

In [ ]:
#확정분 청구분 구분 지침
_CONFIRMED_EXPECTED_CLARIFICATION_PROMPT = """
  목표: 문서 텍스트만으로 확정분/청구분/검토필요/거래없음을 "필드(컬럼 값)" 단위로 안정적으로 분리한다.

  핵심 원칙
  - (R1) 분류 단위는 "필드(컬럼 값)"이다. (행/펀드 단위 아님)
  - (R2) 근거는 반드시 "해당 필드가 속한 컬럼그룹/섹션" 범위로만 서술한다.
  - (R3) 라벨링(확정/청구/검토/거래없음)과 출력(정규화/적재)을 분리한다.
  - (R4) 문서에 없는 값을 생성/치환/역할변경하지 않는다. (공백→0 금지, 작성일→결제일 치환 금지, 검수일 등 임의 생성 금지)
  - (R4-2) raw_value는 원문 그대로 보존한다. 정정은 canonical_value로만 수행한다.
  - (R5) NAV/Price 부재만으로 청구분을 단정하지 않는다.
  - (R6) 확정/청구 라벨과 품질태그(QualityTag)는 독립이다.
  - (R7) 거래유형(설정/해지 등)은 방향, 확정/청구는 시간/상태다.
  - (R8) 컬럼명 표준화는 display_label로만 하고 raw_label은 내부 추적용으로만 보존한다.

  ------------------------------------------------------------
  -1) RowType 선분리(필수, 최우선)
  - FUND_ROW: 실제 펀드/상품 단위 행(펀드코드/상품코드 식별자 유효)
  - SUMMARY_ROW: 합계/소계/Total/XXX_합계/합계(운용사) 등 집계행
  - META_ROW: 문서 메타데이터 표

  규칙:
  - SUMMARY_ROW, META_ROW는 확정/청구/검토/거래없음 라벨 부여 대상이 아니다.
    → 별도 섹션으로 보관하고 정합성 검증에만 사용한다.

  ------------------------------------------------------------
  0) 전처리: 필드 그룹 분리(필수)
  - IDENT: 식별자(펀드코드/펀드명/운용사/수탁사 등) — 라벨 부여 대상 아님(키)
  - DATE: 거래/처리/결제/기준/가격일 등
  - PRICE: NAV/기준가격/단가/환율 등
  - TXN_EXEC: 거래 실행/요청 좌수·금액 (설정/해지/매수/매도/유입/유출 등)
  - TXN_PENDING: 미처리/대기/보류/미결제/미정산/Pending/Unprocessed/TBD 좌수·금액
  - BAL: 전일/전후/잔여/변경후 잔고
  - DERIVED: 순유입/증감/합산/비율 등 파생 지표

  ------------------------------------------------------------
  0-1) 값(value) 처리 정책(필수: 정정값 유지)
  - raw_value: 원문/추출값 그대로(공백 포함) 보존
  - canonical_value: 의미 불변 입력/추출 오류를 정정한 값 → 기본 출력/적재 값
  - display_value: 화면 표시용(기본 canonical_value)

  허용 정정(의미 불변)
  - 앞/뒤 공백 제거, 연속 공백 1칸화, 전각/특수공백 통일, 제어문자 제거
  → CorrectionTag=VALUE_WHITESPACE_NORMALIZED
  ※ raw_value는 그대로 두고 canonical_value만 바꾼다.

  ------------------------------------------------------------
  0-2) 거래 존재 여부 게이트(펀드행에만 적용)  ★설정/해지 거래없음 분리 포함★

  정의:
  - "설정과 해지 모두 거래 없음" = 문서에서 설정과 해지에 해당하는 TXN_EXEC 필드(좌수/금액)가 모두 0
    (일반화: 문서의 “방향성 거래 필드” 전체가 0인 상태)

  (1) 거래없음(명시적)
  - TXN_EXEC과 TXN_PENDING이 모두 '-'/공란/No order/없음
  → 라벨: 거래없음

  (2) 설정과 해지 모두 거래없음(0-only)  ★요구사항 반영: 따로 분류★
  - (설정 관련 TXN_EXEC) 모든 값 = 0
  - AND (해지 관련 TXN_EXEC) 모든 값 = 0
  - AND TXN_PENDING 모든 값 = 0
  - AND (BAL 전후가 존재하면) 전후 동일(변화 없음)
  - AND A(미확정/향후) 신호 없음
  → 라벨: 거래없음
  → QualityTag=NO_SUB_REDEEM_AND_NO_PENDING

  (3) 대기 거래 존재(혼재 가능)
  - TXN_PENDING 중 하나라도 ≠ 0
  → 해당 TXN_PENDING 필드 라벨: 청구분(A 우선)
  → TXN_EXEC/BAL/DATE/PRICE는 1) 규칙으로 계속 판정

  (4) 그 외 → 1) 판정 규칙으로 진행

  강제 규칙:
  - 거래없음으로 판정된 펀드는 "거래없음 섹션"에만 출력(다른 섹션 혼입 금지)

  ------------------------------------------------------------
  1) 판정 규칙(우선순위: A > B > C > E > E2 > D)

  A) 미확정/향후/대기 신호가 붙은 "해당 필드" → 청구분(최우선)
  - 예정/예상/Forecast/Expected/Scheduled/Plan, Pending/TBD/To be confirmed
  - 추후정산/정산손익 미반영/변동 가능/after valuation
  - 컬럼/라벨 자체가 미처리/대기/보류/미정산/미결제/Pending/Unprocessed/TBD 이면:
    - 값 ≠ 0 : 청구분
    - 값 = 0 : 청구 없음(0) → 출력 생략 권장 + QualityTag=PENDING_ZERO

  B) 확정/정산/결제완료/처리완료 신호가 명시 → 확정분

  E) 확정 판단(강)
  - PRICE + 관련 DATE + TXN_EXEC + (A 신호 없음)

  E2) 확정 판단(강)
  - TXN_EXEC + BAL + DATE 롤포워드 일관성 + (A 신호 없음)
  - 단, TXN_PENDING 필드는 E/E2로 확정 처리하지 않는다(A 우선)

  혼재 처리(필수)
  - 동일 펀드에 TXN_EXEC와 TXN_PENDING이 함께 존재할 수 있다.
    - TXN_EXEC 필드: 확정분(E/E2/B)
    - TXN_PENDING(≠0) 필드: 청구분(A)
    - BAL:
      - BAL 롤포워드가 TXN_EXEC만으로 성립하면 확정분
      - 성립하지 않으면 검토필요 + QualityTag=PENDING_BAL_AMBIGUOUS

  D) 애매/품질 이슈 → 검토필요

  ------------------------------------------------------------
  2) 출력 템플릿(강제; R1/R3 준수)

  - META_ROW / SUMMARY_ROW: 별도 섹션(라벨 없음)
  - 거래없음 섹션: NO_SUB_REDEEM_AND_NO_PENDING 판정 펀드만 출력(다른 섹션 혼입 금지)
  - 확정분 섹션: 라벨=확정분인 "필드만" 출력(TXN_EXEC/BAL/DATE/PRICE)
  - 청구분 섹션: 라벨=청구분인 "필드만" 출력(권장: TXN_PENDING 중 ≠0만)
  - 검토필요 섹션: 라벨=검토필요인 "필드만" 출력

  ------------------------------------------------------------
  3) 합계/정합성 검증
  - 합계 행은 SUMMARY_ROW로만 사용(정합성 검증용), 분류 라벨 대상 아님
  - 같은 지표끼리만 합계 검증(확정끼리/청구끼리/거래없음 제외)

"""


# markdown으로 정리된 변액일임펀드 설정/해지 지시서 문서에서 확정분과 청구분을 구분하는 기준에 따라 확정분과 청구분으로 데이터를 구분하는 프롬프트
def get_prompt_confirmed_expected_clarification(instruction_markdown: str):
  prompt_text = f"""
   아래의 제공된 문서는 수익자가 보내온 변액일임펀드 설정/해지 지시 내용을 markdown 형식으로 작성한 문서입니다.

   제공된 문서의 내용을 아래의 확정분과 청구분 분류 기준에 따라 확정분과 청구분 그리고 거래없음으로 분류하여 정리하세요. 

   검수 지침에 따라 정리한 내용을 검수 하세요.

   검수가 완료되면 정리된 결과를 아래의 출력 형식에 따라 markdown 형식으로 출력하세요.
   


   # 확정분과 청구분 분류 기준

   {_CONFIRMED_EXPECTED_CLARIFICATION_PROMPT}
   

   # 검수 지침 (반드시 수행)

   - 누락된 데이터가 있는지 확인한다.

   - 확정분과 청구분 분류 기준에 따라 정확하게 분류되었는지 확인한다.

   - 설정좌수와 설정금액 그리고 해지좌수와 해지금액이 전부 0인 거래는 거래 없음으로 분류되었는지 확인한다.(설정좌수, 설정금액, 해지좌수, 해지금액 중에서 하나라도 0보다 큰 항목이 있으면 거래가 있음으로 판단)

   - 거래 없음으로 분류된 데이터의 설정(좌수/금액)과 해지(좌수/금액)가 모두 0인지 확인한다.

   - 제공된 문서의 전체 거래 집계 현황과 정리된 결과가 일치하는지 확인한다.

   - 제공된 문서의 전체 펀드 개수, 펀드코드 개수, 거래 건수 집계 현황과 정리된 결과가 일치하는지 확인한다.

   - 거래없음, 확정분, 청구분 펀드 개수, 거래 건수 집계 현황과 분류된 거래없음 데이터, 확정분 데이터, 청구분 데이터 개수가 일치하는지 확인한다.

   - 집계한 거래없음 펀드 개수와 분류된 거래없음 데이터 개수가 일치하는지 확인한다.

   - 집계한 확정분 펀드 개수와 분류된 확정분 데이터 개수가 일치하는지 확인한다.

   - 집계한 청구분 펀드 개수와 분류된 청구분 데이터 개수가 일치하는지 확인한다.

   - Markdown 코드의 오류 여부를 검수하여 오류가 발견되면 수정한다.

   - 검수 결과에서 오류가 발견되면 반드시 모든 오류를 수정한 뒤 결과를 출력한다.


   # 제공된 문서

   {instruction_markdown}


   # 출력 형식 (반드시 markdown 형식으로 순서대로 출력)

   1. 매입통보일 / 환매신청일

   2. 전체 거래 집계 현황 테이블

   3. 전체 편드 코드 개수, 거래 개수 집계 현황 테이블
    3-1. 거래없음 펀드 코드 개수, 거래 개수 집계 현황
    3-2. 확정분 펀드 코드 개수, 거래 개수 집계 현황
    3-3. 청구분 펀드 코드 개수, 거래 개수 집계 현황 
   
   4. 거래없음 데이터 테이블

   5. 확정분 데이터 테이블

   6. 청구분 데이터 테이블   

   7. 검수 결과 보고서


   # IMPORTANT: 검수 결과 보고서를 확인하고 오류가 있으면 반드시 수정하고 다시 출력하세요!

  """
  return prompt_text


  # markdown으로 정리된 변액일임펀드 설정/해지 지시서 문서에서 확정분과 청구분을 구분하는 기준에 따라 확정분과 청구분으로 데이터를 구분하는 프롬프트
def backup_get_prompt_confirmed_expected_clarification(instruction_markdown: str):
  prompt_text = f"""
   아래의 제공된 문서는 수익자가 보내온 변액일임펀드 설정/해지 지시 내용을 markdown 형식으로 작성한 문서입니다.

   아래의 확정분과 청구분 구분 기준에 따라 데이터를 확정분과 청구분으로 구분하여 정리하세요. 

   지침에 따라 정리한 내용을 markdown 형식으로 작성하세요.
   


   # 확정분과 청구분 구분 기준

   {_CONFIRMED_EXPECTED_CLARIFICATION_PROMPT}



   # 제공된 문서

   {instruction_markdown}
   
   

   # 반드시 지켜야 할 중요 지침

   - 문서 전체 내용을 분석하세요.

   - 문서에서 확정분과 청구분을 분류하는 기준에 따라 확정분과 청구분으로 설정/해지 데이터를 분류하세요.

   - 거래가 없는 종목(펀드)은 거래 없음으로 분류하세요.

   - 확정분에 해당하는 데이터를 수집하여 테이블 형식으로 정리하세요.

   - 확정분으로 판단한 근거를 설명하세요.

   - 청구분에 해당하는 데이터를 수집하여 테이블 형식으로 정리하세요.

   - 청구분으로 판단한 근거를 설명하세요.

   - 확정분에 해당하는 데이터가 없으면 확정분 데이터가 없음을 출력하세요.

   - 확정분에 해당하는 데이터가 없으면 확정분 데이터 없음과 판단한 근거를 설명하세요.

   - 청구분에 해당하는 데이터가 없으면 청구분 데이터가 없음을 출력하세요.

   - 청구분에 해당하는 데이터가 없으면 청구분 데이터 없음과 판단한 근거를 설명하세요.

   - 통합 또는 요약하지 말고, 종목(펀드)명과 펀드코드 단위로 data를 정리하세요.

   - 누락된 필드가 있는지 확인하세요. 누락된 필드가 있으면 추가하세요.

   - 누락된 데이터가 있는지 확인하세요. 누락된 데이터가 있으면 추가하세요.

   - 금액과 합계 데이터가 있는지 확인하세요. 합계가 정확한지 검증하세요.

   - 모든 수집 결과의 테이블에서 데이터가 인접한 컬럼에 중복되거나 병합되어 작성되어 있는지 확인하세요. 중복 또는 병합되어 있으면 수정하세요.

   - 모든 수집 결과에서 단어 또는 숫자를 임의로 제거하지 말고 있는 그대로 출력하세요.

   - 모든 수집 결과에서 단어 또는 숫자를 임의로 요약하지 말고 있는 그대로 출력하세요.

   - 모든 수집 결과에서 단어 또는 숫자를 임의로 새로 작성하지 말고 있는 그대로 출력하세요.

   - 단어의 의미를 분석하고 맥락을 통해 공백, 띄어쓰기, 줄바꿈 오류가 존재하는지 확인하고 오류가 있으면 단어의 의미와 맥락에 맞도록 수정하세요.

   - 종목명에서 공백, 띄어쓰기, 줄바꿈 오류가 자주 발생합니다. 종목명에서 공백, 띄어쓰기, 줄바꿈 오류가 존재하는지 확인하고 오류가 있으면 단어의 의미와 맥락에 맞도록 수정하세요.

   - 종목명에서 약어를 사용하는지 판단하여 약어를 유지하세요.

   - 종목명(펀드명)과 펀드코드가 정확한지 반드시 확인하세요.

   - 모든 수집 결과의 메타데이터(컬럼, 필드)를 빠짐없이 분석하여 의미와 기능을 설명하세요.

   - 원문을 번역하지 말고 원문 그대로 출력하세요.


   # 출력 데이터 검수(반드시 수행)

   - 지침에 따라 정확하게 작성되었는지 확인한다.

   - 확정분과 청구분을 분류하는 기준에 따라 정확히 추출되었는지 확인한다.

   - 거래가 없는 종목(펀드)은 거래 없음으로 분류되었는지 확인한다.

   - 확정분과 청구분에 대한 필드와 텍스트가 정확히 추출되었는지 확인한다.

   - 모든 항목에서 종목명(펀드명)과 펀드코드가 정확하게 작성되었는지 재확인한다.

   - 확정분에 해당하는 데이터가 없으면 데이터가 없음으로 출력되었는지 확인한다.

   - 확정분에 해당하는 데이터가 없으면 데이터 없음과 근거만 출력되었는지 확인한다.

   - 청구분에 해당하는 데이터가 없으면 데이터가 없음으로 출력되었는지 확인한다.

   - 청구분에 해당하는 데이터가 없으면 데이터 없음과 근거만 출력되었는지 확인한다.

   - Markdown 코드의 오류 여부를 검수하여 오류가 발견되면 수정한다.

   - 오류가 있으면 수정한 뒤 검수 결과를 작성한다.


   # 출력 형식(반드시 순서대로 출력)
   1. 확정분 데이터 테이블
   2. 청구분 데이터 테이블
   3. 거래없음 데이터 테이블
   4. 검수 결과 보고서    
  """
  return prompt_text



In [ ]:


# markdown으로 확정분과 청구분을 구분하여 정리된 문서에서 확정분을 수집하는 프롬프트
def get_prompt_confirmed_data(clarification_markdown: str):
  prompt_text = f"""
   
  """
  return prompt_text


# markdown으로 확정분과 청구분을 구분하여 정리된 문서에서 청구분을 수집하는 프롬프트
def get_prompt_expected_data(clarification_markdown: str):
  prompt_text = f"""

  """
  return prompt_text




# Util

In [ ]:
from pathlib import Path
from typing import List

def get_file_path(file_path: str) -> str:
    """
    파일 경로를 절대 경로로 변환하는 함수
    
    Args:
        file_path: 상대 경로 또는 절대 경로
    """
    # 파일 경로 확인 및 절대 경로로 변환
    file_path = Path(file_path)
    if not file_path.is_absolute():
        # 노트북 위치 기준 상대 경로 처리
        # 노트북은 asset_ai_portal/tests 폴더에 있고, documents는 20_code_test 루트에 있음
        current_dir = Path.cwd()
        
        # asset_ai_portal/tests에서 실행 중이면 상위로 두 번 이동 (20_code_test 루트)
        if current_dir.name == 'tests' and current_dir.parent.name == 'asset_ai_portal':
            project_root = current_dir.parent.parent  # tests -> asset_ai_portal -> 20_code_test
        elif current_dir.name == 'asset_ai_portal':
            project_root = current_dir.parent  # asset_ai_portal -> 20_code_test
        else:
            # 20_code_test에서 실행 중이면 그대로 사용
            project_root = current_dir
        
        file_path = project_root / file_path
    
    if not file_path.exists():
        raise FileNotFoundError(f"PDF 파일을 찾을 수 없습니다: {file_path}")
    
    return str(file_path)


def table_to_markdown(table: List[List]) -> str:
    """
    표 데이터를 마크다운 테이블 형식으로 변환하는 헬퍼 함수
    
    Args:
        table: 2차원 리스트 형태의 표 데이터
    
    Returns:
        마크다운 테이블 문자열
    """
    if not table or len(table) == 0:
        return ""
    
    # 빈 셀을 빈 문자열로 변환
    def clean_cell(cell):
        if cell is None:
            return ""
        return str(cell).strip()
    
    # 표 데이터 정리
    cleaned_table = [[clean_cell(cell) for cell in row] for row in table]
    
    # 최대 컬럼 수 확인
    max_cols = max(len(row) for row in cleaned_table) if cleaned_table else 0
    
    # 모든 행을 동일한 컬럼 수로 맞춤
    normalized_table = []
    for row in cleaned_table:
        normalized_row = row + [""] * (max_cols - len(row))
        normalized_table.append(normalized_row)
    
    if not normalized_table:
        return ""
    
    markdown_lines = []
    
    # 헤더 행 (첫 번째 행)
    header = normalized_table[0]
    markdown_lines.append("| " + " | ".join(header) + " |")
    
    # 구분선
    markdown_lines.append("| " + " | ".join(["---"] * len(header)) + " |")
    
    # 데이터 행들
    for row in normalized_table[1:]:
        markdown_lines.append("| " + " | ".join(row) + " |")
    
    return "\n".join(markdown_lines)


# from IPython.display import Markdown, display

# def display_markdown(response_text):
#     # LLM 응답을 마크다운 형식으로 보기 좋게 표시
#     if 'response' in locals():
#         display(Markdown(response.content))
        
#         # 추가 정보 (토큰 사용량 등)를 표시
#         if hasattr(response, 'response_metadata') and response.response_metadata:
#             metadata = response.response_metadata
#             if 'token_usage' in metadata:
#                 print("\n---")
#                 print("**토큰 사용량:**")
#                 print(f"- 입력 토큰: {metadata['token_usage'].get('prompt_tokens', 'N/A')}")
#                 print(f"- 출력 토큰: {metadata['token_usage'].get('completion_tokens', 'N/A')}")
#                 print(f"- 총 토큰: {metadata['token_usage'].get('total_tokens', 'N/A')}")
#     else:
#         print("⚠️ 'response' 변수를 찾을 수 없습니다. 먼저 LLM을 호출해주세요.")


def get_last_ai_message(test_result):
    """
    _test_result의 'messages' 배열에서 마지막 AIMessage를 가져와서 반환하는 함수
    
    Args:
        test_result: agent 실행 결과 딕셔너리 (messages 키 포함)
    
    Returns:
        마지막 AIMessage 객체 또는 None
    """
    if not test_result or "messages" not in test_result:
        return None
    
    messages = test_result["messages"]
    
    # 역순으로 순회하여 첫 번째 AIMessage 찾기
    for message in reversed(messages):
        if isinstance(message, AIMessage):
            return message
    
    return None

In [ ]:
import json

from rich.console import Console
from rich.panel import Panel
from rich.text import Text

console = Console()

def format_message_content(message):
    """Convert message content to displayable string."""
    parts = []
    tool_calls_processed = False

    # Handle main content
    if isinstance(message.content, str):
        parts.append(message.content)
    elif isinstance(message.content, list):
        # Handle complex content like tool calls (Anthropic format)
        for item in message.content:
            if item.get("type") == "text":
                parts.append(item["text"])
            elif item.get("type") == "tool_use":
                parts.append(f"\n🔧 Tool Call: {item['name']}")
                parts.append(f"   Args: {json.dumps(item['input'], indent=2, ensure_ascii=False)}")
                parts.append(f"   ID: {item.get('id', 'N/A')}")
                tool_calls_processed = True
    else:
        parts.append(str(message.content))

    # Handle tool calls attached to the message (OpenAI format) - only if not already processed
    if (
        not tool_calls_processed
        and hasattr(message, "tool_calls")
        and message.tool_calls
    ):
        for tool_call in message.tool_calls:
            parts.append(f"\n🔧 Tool Call: {tool_call['name']}")
            parts.append(f"   Args: {json.dumps(tool_call['args'], indent=2, ensure_ascii=False)}")
            parts.append(f"   ID: {tool_call['id']}")

    return "\n".join(parts)


def format_messages(messages):
    """Format and display a list of messages with Rich formatting."""
    for m in messages:
        msg_type = m.__class__.__name__.replace("Message", "")
        content = format_message_content(m)

        if msg_type == "Human":
            console.print(Panel(content, title="🧑 Human", border_style="blue"))
        elif msg_type == "Ai":
            console.print(Panel(content, title="🤖 Assistant", border_style="green"))
        elif msg_type == "Tool":
            console.print(Panel(content, title="🔧 Tool Output", border_style="yellow"))
        else:
            console.print(Panel(content, title=f"📝 {msg_type}", border_style="white"))


def format_message(messages):
    """Alias for format_messages for backward compatibility."""
    return format_messages(messages)

# PDF 파서

### pdfplumber

In [ ]:
# pdfplumber 파서
def extract_pdf_with_pdfplumber(pdf_path: str, password: str = None) -> str:
    """
    pdfplumber를 사용하여 PDF 파일을 마크다운 형식으로 변환하는 함수
    
    pdfplumber는 PDF 파일을 텍스트 데이터로 추출하는 라이브러리로, 표, 이미지, 레이아웃 등을 잘 보존합니다.
    암호화된 PDF와 암호화되지 않은 PDF 모두 처리할 수 있습니다.
    """
    try:
        import pdfplumber
    except ImportError:
        raise ImportError(
            "PDF를 처리하기 위해 pdfplumber가 필요합니다.\n"
            "설치 명령: pip install pdfplumber"
        )
    
    markdown_parts = []
    
    try:
        pdf_path = get_file_path(pdf_path)
        # password가 있으면 암호화된 PDF로 처리, 없으면 암호화되지 않은 PDF로 처리
        pdf_kwargs = {"password": password} if password else {}
        
        with pdfplumber.open(pdf_path, **pdf_kwargs) as pdf:
            for page_num, page in enumerate(pdf.pages, 1):
                page_content = []
                
                # 표 추출 (표가 있으면 먼저 표를 추출)
                tables = page.extract_tables()
                if tables:
                    for table_idx, table in enumerate(tables):
                        if table:
                            markdown_table = table_to_markdown(table)
                            if markdown_table:
                                page_content.append(markdown_table)
                                page_content.append("")  # 표 다음에 빈 줄 추가
                
                # 텍스트 추출
                text = page.extract_text()
                if text:
                    page_content.append(text)
                
                if page_content:
                    markdown_parts.append("\n".join(page_content))
        
        return "\n\n".join(markdown_parts) if markdown_parts else ""
        
    except Exception as e:
        # 암호화 관련 오류인지 확인
        error_msg = str(e).lower()
        if 'password' in error_msg or 'encrypted' in error_msg or 'incorrect password' in error_msg:
            raise ValueError(f"PDF 암호가 올바르지 않거나 암호화된 PDF를 읽을 수 없습니다: {e}")
        raise


### pikepdf

In [ ]:
from datetime import datetime
from pathlib import Path
import pikepdf

def _decrypt_to_temp_pdf(src_pdf: str, password: str) -> str:
    """pikepdf로 PDF를 열어 '복호화된' PDF를 원본과 동일한 위치에 타임스탬프를 붙인 파일명으로 저장 후 경로 반환"""

    src_pdf = Path(src_pdf)
    
    # 원본 PDF와 동일한 디렉토리에 저장
    parent_dir = src_pdf.parent
    stem = src_pdf.stem
    suffix = src_pdf.suffix
    
    # 타임스탬프 생성 (YYYYMMDD_HHMMSS 형식)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    new_filename = f"{stem}_{timestamp}{suffix}"
    decrypted_pdf = parent_dir / new_filename

    if password:
        with pikepdf.open(str(src_pdf), password=password) as pdf:
            pdf.save(str(decrypted_pdf))  # 저장 시 기본적으로 암호가 제거된 형태로 저장됨(열기 암호 제거 목적)
    else:
        with pikepdf.open(str(src_pdf)) as pdf:
            pdf.save(str(decrypted_pdf))  # 저장 시 기본적으로 암호가 제거된 형태로 저장됨(열기 암호 제거 목적)
    return decrypted_pdf

### docling

In [ ]:
from typing import Optional
from pathlib import Path

from docling.datamodel.base_models import InputFormat
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions, TableFormerMode
from docling.datamodel.accelerator_options import AcceleratorOptions

# ✅ 표 구조 복원에 유리한 권장 백엔드 (기본값이기도 함) :contentReference[oaicite:5]{index=5}
from docling.backend.docling_parse_v4_backend import DoclingParseV4DocumentBackend

try:
    from docling.document_converter import PdfBackendOptions
except ImportError:
    from docling.datamodel.base_models import PdfBackendOptions

from docling_core.types.doc.document import ContentLayer  # :contentReference[oaicite:3]{index=3}

def extract_text_from_pdf_with_docling_nopassword(pdf_path: str) -> str:
    pdf_path = get_file_path(pdf_path)

    def _run(do_cell_matching: bool) -> str:
        pipeline = PdfPipelineOptions(
            do_ocr=False,
            do_table_structure=True,
            do_picture_classification=False,
            do_picture_description=False,
            generate_page_images=False,
            images_scale=2.0,  # 레이아웃/테이블 크롭 품질에 도움될 수 있음 :contentReference[oaicite:6]{index=6}
        )

        # ✅ 표 구조 품질 우선
        pipeline.table_structure_options.mode = TableFormerMode.ACCURATE  # :contentReference[oaicite:7]{index=7}
        pipeline.table_structure_options.do_cell_matching = do_cell_matching  # :contentReference[oaicite:8]{index=8}

        # ✅ 표 구조 목적이면 force_backend_text는 끄는 쪽이 안전
        pipeline.force_backend_text = False  # :contentReference[oaicite:9]{index=9}

        pipeline.accelerator_options = AcceleratorOptions(device="cpu")

        # backend_opts = PdfBackendOptions(password=password) if password else None

        converter = DocumentConverter(
            allowed_formats=[InputFormat.PDF],
            format_options={
                InputFormat.PDF: PdfFormatOption(
                    pipeline_options=pipeline,
                    backend=DoclingParseV4DocumentBackend,
                    # backend_options=backend_opts,
                )
            },
        )

        result = converter.convert(pdf_path)
        # md = result.document.export_to_markdown()
        md = result.document.export_to_markdown(
            included_content_layers={ContentLayer.BODY},
            # page_break_placeholder="\n\n<!-- pagebreak -->\n\n",                 # (선택) 페이지 경계 표시 :contentReference[oaicite:5]{index=5}
        )

        # “테이블이 전혀 안 잡혔는지” 빠른 휴리스틱 (파이프 문자 기반)
        # 필요하면 여기서 result.document에서 TableItem 개수를 세는 방식으로 더 정확히 판단 가능 :contentReference[oaicite:10]{index=10}
        return md

    try:
        md = _run(do_cell_matching=True)    
        return md
    except Exception as e:
        print(e)
        return ""

    # 1차: 기본(셀 매칭 True)
    # md = _run(do_cell_matching=True)
    # if "|---" in md or "| ---" in md:
    #     print("do_cell_matching=True")
    #     return md

    # 2차: 셀 매칭 False (borderless/매칭 실패 케이스에 유리) :contentReference[oaicite:11]{index=11}
    # md2 = _run(do_cell_matching=False)
    # print("do_cell_matching=False")
    # return md2

def extract_pdf_with_docling(pdf_path: str, password: str = None) -> str:
    if password:
        # _decrypt_to_temp_pdf는 문자열 경로를 반환
        pdf_path = _decrypt_to_temp_pdf(pdf_path, password)
        pdf_text = extract_text_from_pdf_with_docling_nopassword(pdf_path)

        # 임시파일 삭제
        # os.remove(pdf_path)
        
        return pdf_text
    else:
        return extract_text_from_pdf_with_docling_nopassword(pdf_path)
    


# Excel 파서

In [ ]:
# excel loader

import os
from pathlib import Path
from typing import Dict, List

def extract_text_from_excel(excel_path: str) -> Dict[str, str]:
    """
    Excel 파일(.xlsx, .xls)에서 모든 시트의 텍스트를 추출하는 함수
    
    Args:
        excel_path: Excel 파일 경로 (상대 경로 또는 절대 경로)
    
    Returns:
        시트 이름을 키로 하고 추출된 텍스트를 값으로 하는 딕셔너리
    
    Raises:
        FileNotFoundError: Excel 파일을 찾을 수 없을 때
        ImportError: 필요한 Excel 라이브러리가 설치되지 않았을 때
    """
    # 파일 경로 확인 및 절대 경로로 변환    
    excel_path = get_file_path(excel_path)

    print(f"excel_path: {excel_path}")  
    
    # 파일 확장자 확인 (Path 객체로 변환)
    excel_path_obj = Path(excel_path)
    file_ext = excel_path_obj.suffix.lower()
    
    # 여러 Excel 라이브러리 시도 (우선순위 순)
    # 1. pandas + openpyxl/xlrd (가장 편리함)
    try:
        import pandas as pd
        
        # 모든 시트 읽기
        if file_ext == '.xlsx':
            excel_file = pd.ExcelFile(str(excel_path), engine='openpyxl')
        elif file_ext == '.xls':
            excel_file = pd.ExcelFile(str(excel_path), engine='xlrd')
        else:
            # 자동 감지
            excel_file = pd.ExcelFile(str(excel_path))
        
        sheets_text = {}
        for sheet_name in excel_file.sheet_names:
            df = pd.read_excel(excel_file, sheet_name=sheet_name)
            # DataFrame을 텍스트로 변환
            text_parts = []
            # 헤더 포함하여 모든 셀의 값을 문자열로 변환
            for idx, row in df.iterrows():
                row_values = [str(val) if pd.notna(val) else '' for val in row.values]
                text_parts.append(' | '.join(row_values))
            
            sheets_text[sheet_name] = '\n'.join(text_parts)

        print("using pandas")
        return sheets_text
    except ImportError as e:
        if 'pandas' in str(e):
            pass  # pandas가 없으면 다음 방법 시도
        elif 'openpyxl' in str(e) or 'xlrd' in str(e):
            # pandas는 있지만 엔진이 없는 경우
            raise ImportError(
                f"Excel 파일을 읽기 위한 엔진이 필요합니다.\n"
                f".xlsx 파일: pip install openpyxl\n"
                f".xls 파일: pip install xlrd"
            )
        else:
            raise
    
    # 2. openpyxl (xlsx 파일용)
    if file_ext == '.xlsx':
        try:
            from openpyxl import load_workbook
            
            workbook = load_workbook(str(excel_path), data_only=True)
            sheets_text = {}
            
            for sheet_name in workbook.sheetnames:
                sheet = workbook[sheet_name]
                text_parts = []
                
                for row in sheet.iter_rows(values_only=True):
                    row_values = [str(val) if val is not None else '' for val in row]
                    text_parts.append(' | '.join(row_values))
                
                sheets_text[sheet_name] = '\n'.join(text_parts)
            
            print("using openpyxl")
            return sheets_text
        except ImportError:
            pass
    
    # 3. xlrd (xls 파일용)
    if file_ext == '.xls':
        try:
            import xlrd
            
            workbook = xlrd.open_workbook(str(excel_path))
            sheets_text = {}
            
            for sheet_name in workbook.sheet_names():
                sheet = workbook.sheet_by_name(sheet_name)
                text_parts = []
                
                for row_idx in range(sheet.nrows):
                    row_values = [str(sheet.cell_value(row_idx, col_idx)) 
                                 for col_idx in range(sheet.ncols)]
                    text_parts.append(' | '.join(row_values))
                
                sheets_text[sheet_name] = '\n'.join(text_parts)
            
            print("using xlrd")
            return sheets_text
        except ImportError:
            pass
    
    # 모든 라이브러리가 없으면 에러
    raise ImportError(
        "Excel 텍스트 추출을 위한 라이브러리가 설치되지 않았습니다.\n"
        "다음 중 하나를 설치해주세요:\n"
        "  - pandas + openpyxl (권장): pip install pandas openpyxl\n"
        "  - pandas + xlrd (.xls 파일용): pip install pandas xlrd\n"
        "  - openpyxl (.xlsx 파일용): pip install openpyxl\n"
        "  - xlrd (.xls 파일용): pip install xlrd"
    )


def parser_excel(excel_path: str) -> str:
    """
    Excel 파일의 모든 시트 텍스트를 하나의 문자열로 반환하는 편의 함수
    
    Args:
        excel_path: Excel 파일 경로
    
    Returns:
        모든 시트의 텍스트를 합친 문자열
    """
    sheets_dict = extract_text_from_excel(excel_path)
    
    result_parts = []
    for sheet_name, sheet_text in sheets_dict.items():
        result_parts.append(f"=== 시트: {sheet_name} ===")
        result_parts.append(sheet_text)
        result_parts.append("")  # 빈 줄 추가
    
    return '\n'.join(result_parts)


# document loader

In [ ]:
# Document Loader - 파일 형식에 따라 적절한 함수 호출

from pathlib import Path
from typing import Union

def load_document(file_path: str, password: str = None) -> str:
    """
    파일 형식에 따라 적절한 텍스트 추출 함수를 호출하여 텍스트를 반환하는 통합 함수
    
    지원 형식:
    - PDF: .pdf 파일 (암호화된 PDF 지원)
    - Excel: .xlsx, .xls 파일
    
    Args:
        file_path: 문서 파일 경로 (상대 경로 또는 절대 경로)
        password: PDF 파일이 암호화된 경우 비밀번호 (선택사항)
    
    Returns:
        추출된 텍스트 문자열
        - PDF: 전체 텍스트
        - Excel: 모든 시트의 텍스트를 합친 문자열
    
    Raises:
        FileNotFoundError: 파일을 찾을 수 없을 때
        ValueError: 지원하지 않는 파일 형식일 때 또는 PDF 암호가 틀렸을 때
        ImportError: 필요한 라이브러리가 설치되지 않았을 때
    """
    file_path_obj = Path(file_path)
    file_ext = file_path_obj.suffix.lower()
    
    # 파일 형식에 따라 적절한 함수 호출
    if file_ext == '.pdf':
        # PDF 파일 처리 (암호 전달)
        return extract_pdf_with_pdfplumber(file_path, password)
    
    elif file_ext in ['.xlsx', '.xls']:
        # Excel 파일 처리 - 모든 시트의 텍스트를 하나의 문자열로 반환
        return parser_excel(file_path)
    
    else:
        raise ValueError(
            f"지원하지 않는 파일 형식입니다: {file_ext}\n"
            f"지원 형식: .pdf, .xlsx, .xls"
        )




In [ ]:
# text 추출 테스트

# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/라이나_250826.xlsx"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프(2차)_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프(액티브)_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프(퇴직)_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프(퇴직)_251127_20260105_182813.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/카디프_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/하나생명(액티브)_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/iM라이프_250826.xls"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_overseas_settlement/LS.pdf"

# _password = None
# _password = '345678'

# document_text = load_document(_document_file_path, _password)
# print(document_text)

# LLM 모델 생성

In [ ]:
# LLM 모델 정의

LLM_MODEL = os.getenv("LLM_MODEL")
LLM_BASE_URL=os.getenv("LLM_BASE_URL")
LLM_API_KEY=os.getenv("LLM_API_KEY")

def create_llm_model():

    # vLLM 모델 인스턴스 생성
    llm = init_chat_model(
        "openai:",
        temperature=0.0,
        top_p=0.1,  # top_p는 (0, 1] 범위여야 하므로 0.1로 설정
        base_url=LLM_BASE_URL,
        api_key=LLM_API_KEY
    )
    # llm = init_chat_model(
    #     "openai:gpt-4o",
    #     temperature=0.0,
    #     top_p=0.1,  # top_p는 (0, 1] 범위여야 하므로 0.9로 설정
    # )
    return llm

# LLM Tools - TODO Tools

### State

In [ ]:
from typing import Annotated, Literal, NotRequired
from typing_extensions import TypedDict
from langchain.agents import AgentState


# 복잡한 작업 플로우의 진행 상황 추적을 위한 TODO 항목 구조 정의
class Todo(TypedDict):
    """A structured task item for tracking progress through complex workflows.

    Attributes:
        content: Short, specific description of the task
        status: Current state - pending, in_progress, or completed
    """

    content: str
    status: Literal["pending", "in_progress", "completed"]


# 두 파일 딕셔너리 병합, 오른쪽 값이 우선 적용되는 가상 파일 시스템 업데이트용 reducer 함수
def file_reducer(left, right):
    """Merge two file dictionaries, with right side taking precedence.

    Used as a reducer function for the files field in agent state,
    allowing incremental updates to the virtual file system.

    Args:
        left: Left side dictionary (existing files)
        right: Right side dictionary (new/updated files)

    Returns:
        Merged dictionary with right values overriding left values
    """
    if left is None:
        return right
    elif right is None:
        return left
    else:
        return {**left, **right}


# LangGraph AgentState 상속, TODO 리스트와 가상 파일 시스템 포함한 확장 state 구조 정의
class DeepAgentState(AgentState):
    """Extended agent state that includes task tracking and virtual file system.

    Inherits from LangGraph's AgentState and adds:
    - todos: List of Todo items for task planning and progress tracking
    - files: Virtual file system stored as dict mapping filenames to content
    """

    # 작업 플래닝 및 진행 상황 추적을 위한 Todo 리스트 필드
    todos: NotRequired[list[Todo]]
    # 파일명과 내용 매핑, file_reducer로 병합되는 가상 파일 시스템 필드
    files: Annotated[NotRequired[dict[str, str]], file_reducer]

### prompt - WRITE TODOs Description

복잡한 워크플로에서 진행 상황을 추적하기 위해 구조화된 작업 목록(TODO 리스트)을 생성하고 관리합니다.

사용 시점

조율이 필요한 여러 단계의 작업 또는 단순하지 않은 작업

사용자가 여러 작업을 제공했거나 TODO 리스트를 명시적으로 요청한 경우

별도 지시가 없는 한, 단일하고 사소한 작업에는 사용을 피합니다

구조

여러 개의 TODO 객체(content, status, id)를 포함하는 하나의 리스트를 유지합니다

명확하고 실행 가능한(content) 설명을 사용합니다

status 값은 반드시 다음 중 하나여야 합니다: pending, in_progress, completed

모범 사례

한 번에 in_progress 상태인 작업은 하나만 유지합니다

작업이 완전히 끝나면 즉시 completed로 표시합니다

변경 시에는 항상 업데이트된 전체 리스트를 전송합니다

리스트가 산만해지지 않도록 관련 없는 항목은 제거(정리)합니다

진행 업데이트

작업 상태를 변경하거나 내용을 수정하려면 TodoWrite를 다시 호출합니다

진행 상황을 실시간으로 반영하고, 완료 처리를 한꺼번에 몰아서 하지 않습니다

막힌 경우에는 해당 작업을 in_progress로 유지하고, 막힌 원인을 설명하는 새 작업을 추가합니다

파라미터

todos: content 및 status 필드를 포함한 TODO 항목 리스트

반환

새로운 TODO 리스트로 에이전트 상태를 업데이트합니다

In [ ]:
_WRITE_TODOS_DESCRIPTION="""Create and manage structured task lists for tracking progress through complex workflows.

## When to Use
- Multi-step or non-trivial tasks requiring coordination
- When user provides multiple tasks or explicitly requests todo list  
- Avoid for single, trivial actions unless directed otherwise

## Structure
- Maintain one list containing multiple todo objects (content, status, id)
- Use clear, actionable content descriptions
- Status must be: pending, in_progress, or completed

## Best Practices  
- Only one in_progress task at a time
- Mark completed immediately when task is fully done
- Always send the full updated list when making changes
- Prune irrelevant items to keep list focused

## Progress Updates
- Call TodoWrite again to change task status or edit content
- Reflect real-time progress; don't batch completions  
- If blocked, keep in_progress and add new task describing blocker

## Parameters
- todos: List of TODO items with content and status fields

## Returns
Updates agent state with new todo list."""

### tool - write_todos

`InjectedState`와 `Command`를 활용하여 실제 툴을 구현합니다.

*   **`write_todos`**: 입력받은 TODO 리스트로 State를 업데이트하고, 변경 내역을 `ToolMessage`로 기록

In [ ]:
from typing import Annotated

from langchain_core.messages import ToolMessage
from langchain_core.tools import InjectedToolCallId, tool
from langgraph.prebuilt import InjectedState
from langgraph.types import Command

# write_todos 툴 정의, LLM이 전달한 TODO 리스트를 state에 저장 및 메시지 기록
@tool(description=_WRITE_TODOS_DESCRIPTION,parse_docstring=True)
def write_todos(
    todos: list[Todo], tool_call_id: Annotated[str, InjectedToolCallId]
) -> Command:
    """Create or update the agent's TODO list for task planning and tracking.

    Args:
        todos: List of Todo items with content and status
        tool_call_id: Tool call identifier for message response

    Returns:
        Command to update agent state with new TODO list
    """
    # TODO 리스트와 메시지 업데이트를 위한 Command 객체 반환
    return Command(
        update={
            "todos": todos,
            "messages": [
                ToolMessage(f"Updated todo list to {todos}", tool_call_id=tool_call_id)
            ],
        }
    )

### tool - read_todos

`InjectedState`와 `Command`를 활용하여 실제 툴을 구현합니다.

*   **`read_todos`**: State에서 현재 TODO 리스트를 읽어 포맷팅된 문자열을 `ToolMessage`로 반환 (Command 사용)

In [ ]:
# read_todos 툴 정의, 현재 state의 TODO 리스트를 읽어 포맷된 문자열로 반환
@tool(parse_docstring=True)
def read_todos(
    state: Annotated[DeepAgentState, InjectedState],
    tool_call_id: Annotated[str, InjectedToolCallId],
) -> Command:
    """Read the current TODO list from the agent state.

    This tool allows the agent to retrieve and review the current TODO list
    to stay focused on remaining tasks and track progress through complex workflows.

    Args:
        state: Injected agent state containing the current TODO list
        tool_call_id: Injected tool call identifier for message tracking

    Returns:
        Command to update agent state with ToolMessage containing formatted TODO list
    """
    # state에서 todos 리스트 추출, 없으면 빈 리스트 반환
    todos = state.get("todos", [])
    if not todos:
        # TODO 리스트가 비어 있을 때 안내 메시지 반환
        message_content = "No todos currently in the list."
    else:
        # 현재 TODO 리스트를 번호, 이모지, 상태와 함께 포맷팅하여 문자열로 생성
        result = "Current TODO List:\n"
        for i, todo in enumerate(todos, 1):
            status_emoji = {"pending": "⏳", "in_progress": "🔄", "completed": "✅"}
            emoji = status_emoji.get(todo["status"], "❓")
            result += f"{i}. {emoji} {todo['content']} ({todo['status']})\n"
        message_content = result.strip()

    # Command 객체로 래핑하여 ToolMessage와 함께 반환
    return Command(
        update={
            "messages": [
                ToolMessage(message_content, tool_call_id=tool_call_id)
            ],
        }
    )

### prompt - TODO Usage Instruction

사용자의 요청에 따라:

도구 설명에 따라 사용자 요청이 시작될 때 write_todos 도구를 사용해 TODO를 생성한다.

TODO 하나를 수행(완료)한 뒤에는 read_todos를 사용해 TODO들을 순서대로 읽고, 계획을 다시 상기한다.

지금까지 한 작업과 해당 TODO에 대해 성찰한다.

작업을 완료로 표시하고, 다음 TODO로 진행한다.

모든 TODO를 완료할 때까지 이 과정을 반복한다.

- 중요: 어떤 사용자 요청이든 반드시 TODO 형태의 리서치 계획을 만들고, 위 지침에 따라 리서치를 수행하라.
- 중요: 추적해야 할 TODO 개수를 최소화하기 위해, 리서치 작업을 가능한 한 하나의 TODO로 묶어 작성하라.

In [ ]:
_TODO_USAGE_INSTRUCTIONS = """Based upon the user's request:
1. Use the write_todos tool to create TODO at the start of a user request, per the tool description.
2. After you accomplish a TODO, use the read_todos to read the TODOs in order to remind yourself of the plan. 
3. Reflect on what you've done and the TODO.
4. Mark you task as completed, and proceed to the next TODO.
5. Continue this process until you have completed all TODOs.

IMPORTANT: Always create a research plan of TODOs and conduct research following the above guidelines for ANY user request.
IMPORTANT: Aim to batch research tasks into a *single TODO* in order to minimize the number of TODOs you have to keep track of.
"""

# LLM chain - 변액일임펀드 설정/해지 지시서 여부 검증 체인

In [ ]:
# 문서가 변액일임펀드 설정/해지 지시서인지 확인하는 LLM 노드 생성

from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

# 문서에서 추출한 텍스트가 변액일임펀드 설정/해지 지시서 여부를 판단하는 데이터 모델
class GradeDocument(BaseModel):
    """문서에서 추출한 텍스트가 변액일임펀드 설정/해지 지시서 여부를 판단하기 위한 이진 점수"""

    binary_score: str = Field(description="변액일임펀드 설정/해지 지시서 여부를 오직 'yes' 또는 'no'로 판단합니다.")

def create_chain_retrieval_grader():

    llm = create_llm_model()

    # GradeDocument 데이터 모델을 사용하여 구조화된 출력을 생성하는 LLM
    structured_llm_grader = llm.with_structured_output(GradeDocument)

    # 시스템 프롬프트 정의
    system_prompt = """당신은 자산운용사에서 변액일임펀드 설정/해지 업무를 담당하는 오퍼레이터 입니다.
    수익자가 보낸 문서를 분석하여 변액일임펀드 설정/해지 지시서 인지 아래의 평가 기준을 기반으로 평가 하세요.

    # 변액일임펀드 설정/해지 지시서 평가 기준
    1. 설정/해지에 대한 기준일이 있어야 합니다.
    2. 문서에는 종목(펀드)에 대한 확정분 또는 청구(예상)분이 반드시 있어야 합니다.
    3. 종목(펀드)에 대한 확정분이 있을 경우, 이에 대한 설정(투입) 금액 또는 해지(인출) 금액이 반드시 있어야 합니다.
    4. 종목(펀드)에 대한 확정분이 있을 경우, 이에 대한 설정(투입) 좌수 또는 해지(인출) **좌수**가 반드시 있어야 합니다.
    5. 종목(펀드)에 대한 청구(예상)분이 있을 경우 이에 대한 설정(투입) 금액 또는 해지(인출) 금액이 반드시 있어야 합니다.
    6. 종목(펀드)에 대한 청구(예상)분이 있을 경우 이에 대한 설정(투입) 좌수 또는 해지(인출) **좌수**가 반드시 있어야 합니다.

    # 출력 형식
    문서의 변액일임펀드 설정/해지 지시서 여부를 나타내기 위해 이진 점수인 'yes' 또는 'no'를 부여하세요."""

    # 프롬프트 템플릿 생성
    grade_prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("user", "수익자가 보낸 문서 내용: \n\n {original_text}")
    ])

    # Retrieval 평가기 초기화
    chain_retrieval_grader = grade_prompt | structured_llm_grader
    return chain_retrieval_grader




In [ ]:
# text 추출 태스트

# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/라이나_250826.xlsx"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프(2차)_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프(액티브)_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프(퇴직)_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프(퇴직)_251127_20260105_182813.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/카디프_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/하나생명(액티브)_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/iM라이프_250826.xls"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_overseas_settlement/LS.pdf"

# _password = None
# _password = '345678'

# document_text = load_document(_document_file_path, _password)
# print(document_text)

In [ ]:
# 문서 검증 테스트

# document_text = "test"

# relevance_result = chain_retrieval_grader.invoke({"original_text": document_text})
# print(relevance_result)

# LLM - pdf text to markdown 변환

In [ ]:
def convert_pdf_text_to_markdown(document_text_docling: str, document_text_pdfplumber: str):

    # 메시지 객체 생성
    system_msg = SystemMessage(_SYSTEM_PROMPT)
    human_prompt = get_prompt_pdf_text_to_markdown(document_text_docling, document_text_pdfplumber)
    human_msg = HumanMessage(human_prompt)

    # 채팅 모델과 함께 사용
    messages = [system_msg, human_msg]

    llm = create_llm_model()
    response = llm.invoke(messages)  # AIMessage 반환

    return response

### Agent - with plan tool

In [ ]:
from langchain.agents import create_agent

def convert_pdf_text_to_markdown_using_agent__(document_text_docling: str, document_text_pdfplumber: str):

    tools = [write_todos, read_todos]
    llm = create_llm_model()
    human_prompt = get_prompt_pdf_text_to_markdown(document_text_docling, document_text_pdfplumber)

    agent = create_agent(
        llm,
        tools,
        system_prompt=_TODO_USAGE_INSTRUCTIONS
        + "\n\n"
        + "=" * 80
        + "\n\n"
        + _SYSTEM_PROMPT_ENG,
        state_schema=DeepAgentState,
    )

    response = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": human_prompt
                }
            ],
            "todos": [],
        }
    )

    return response


def convert_pdf_text_to_markdown_using_agent___(document_text_docling: str, document_text_pdfplumber: str):

    tools = [write_todos, read_todos]
    llm = create_llm_model()
    human_prompt = get_prompt_pdf_text_to_markdown(document_text_docling, document_text_pdfplumber)

    user_message = _SYSTEM_PROMPT + "\n\n" + "=" * 80 + "\n\n" + human_prompt

    agent = create_agent(
        llm,
        tools,
        system_prompt=_TODO_USAGE_INSTRUCTIONS,
        state_schema=DeepAgentState,
    )

    response = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": user_message
                }
            ],
        }
    )

    return response

from langchain.agents.middleware import TodoListMiddleware

def convert_pdf_text_to_markdown_using_agent(document_text_docling: str, document_text_pdfplumber: str):

    llm = create_llm_model()
    human_prompt = get_prompt_pdf_text_to_markdown(document_text_docling, document_text_pdfplumber)

    todo_agent = create_agent(
        llm,
        system_prompt=_SYSTEM_PROMPT,
        middleware=[TodoListMiddleware()]
    )

    response = todo_agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": human_prompt
                }
            ],
        }
    )

    return response

# LLM - excel text to markdown

In [ ]:
def convert_excel_text_to_markdown(document_text_excel: str):

    # 메시지 객체 생성
    system_msg = SystemMessage(_SYSTEM_PROMPT)
    human_prompt = get_prompt_text_to_markdown(document_text_excel)
    human_msg = HumanMessage(human_prompt)

    # 채팅 모델과 함께 사용
    messages = [system_msg, human_msg]

    llm = create_llm_model()
    response = llm.invoke(messages)  # AIMessage 반환

    return response

### Agent - with plan tool

In [ ]:
from langchain.agents import create_agent

def convert_excel_text_to_markdown_using_agent_(document_text_excel: str):

    tools = [write_todos, read_todos]
    llm = create_llm_model()
    human_prompt = get_prompt_text_to_markdown(document_text_excel)

    agent = create_agent(
        llm,
        tools,
        system_prompt=_TODO_USAGE_INSTRUCTIONS
        + "\n\n"
        + "=" * 80
        + "\n\n"
        + _SYSTEM_PROMPT_ENG,
        state_schema=DeepAgentState,
    )

    response = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": human_prompt
                }
            ],
        }
    )

    return response


def convert_excel_text_to_markdown_using_agent__(document_text_excel: str):

    tools = [write_todos, read_todos]
    llm = create_llm_model()
    human_prompt = get_prompt_text_to_markdown(document_text_excel)

    user_message = _SYSTEM_PROMPT + "\n\n" + "=" * 80 + "\n\n" + human_prompt

    agent = create_agent(
        llm,
        tools,
        system_prompt=_TODO_USAGE_INSTRUCTIONS,
        state_schema=DeepAgentState,
    )

    response = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": user_message
                }
            ],
        }
    )

    return response


from langchain.agents.middleware import TodoListMiddleware

def convert_excel_text_to_markdown_using_agent(document_text_excel: str):

    llm = create_llm_model()
    human_prompt = get_prompt_text_to_markdown(document_text_excel)

    todo_agent = create_agent(
        llm,
        system_prompt=_SYSTEM_PROMPT,
        middleware=[TodoListMiddleware()]
    )

    response = todo_agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": human_prompt
                }
            ],
        }
    )

    return response

# Agent Test

In [ ]:
# # graph 실행

# # text 추출 태스트

# _test_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/라이나_250826.xlsx"
# # _test_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프_251127.pdf"
# # _test_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프(2차)_251127.pdf"
# # _test_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프(액티브)_251127.pdf"
# # _test_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프(퇴직)_251127.pdf"
# # _test_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/카디프_251127.pdf"
# # _test_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/하나생명(액티브)_251127.pdf"
# # _test_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/iM라이프_250826.xls"
# # _test_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/ABL_250826.xlsx"
# # _test_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/DB_250826.xlsx"
# # _test_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/KB라이프_250826.xls"
# # _test_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/KB라이프(액티브)_250826.xls"
# # _test_file_path = "/Users/bhkim/20_code_test/documents/sample_overseas_settlement/LS.pdf"

# _test_password = None
# # _test_password = '345678'

# from pathlib import Path

# # file_path에서 파일명과 확장자 추출    
# _path_obj = Path(_test_file_path)
# _test_file_name = _path_obj.name  # 파일명 (확장자 포함)
# _test_file_ext = _path_obj.suffix  # 확장자 (점 포함, 예: .pdf)

# if _test_file_ext == ".pdf":
#     # 문서 파싱 노드 라우팅 - 병렬 실행
#     _pdf_text_docling = extract_pdf_with_docling(_test_file_path, _test_password)
#     _pdf_text_pdfplumber = extract_pdf_with_pdfplumber(_test_file_path, _test_password)
#     _test_result = convert_pdf_text_to_markdown_using_agent(_pdf_text_docling, _pdf_text_pdfplumber)
# elif _test_file_ext == ".xlsx" or _test_file_ext == ".xls":
#     # 엑셀 문서 파싱 노드 라우팅
#     _excel_text = parser_excel(_test_file_path)
#     _test_result = convert_excel_text_to_markdown_using_agent(_excel_text)

# if _test_result:
#     print(f"   test_result : {_test_result}")
#     format_messages(_test_result["messages"])
# else:
#     print("   test_result is None")


In [ ]:
# agent 결과 출력 함수
from langchain.messages import AIMessage
from rich.console import Console
from rich.panel import Panel
from rich.text import Text

from IPython.display import Markdown, display

console = Console()

def display_last_ai_message(test_result):
    """
    _test_result의 'messages' 배열에서 마지막 AIMessage를 가져와서 화면에 출력하는 함수
    
    Args:
        test_result: agent 실행 결과 딕셔너리 (messages 키 포함)
    """
    if not test_result:
        print("test_result가 None입니다.")
        return
    
    if "messages" not in test_result:
        print("test_result에 'messages' 키가 없습니다.")
        return
    
    last_ai_message = get_last_ai_message(test_result)
    
    if last_ai_message is None:
        print("AIMessage를 찾을 수 없습니다.")
        return
    
    # AIMessage 내용 원문 그대로 출력
    content = last_ai_message.content if hasattr(last_ai_message, 'content') else str(last_ai_message)
    # print(content)
    return content

# 함수 실행
# if '_test_result' in globals() and _test_result:
#     _content = display_last_ai_message(_test_result)
#     display(Markdown(_content))
# else:
#     print("_test_result 변수가 없거나 None입니다.")

# LLM - pdf to markdown 정리내용 검수

In [ ]:
# pdf to markdown으로 정리된 지시서를 받아 확정분과 청구분으로 구분하여 데이터 정리
def validate_pdf_instruction_markdown(instruction_markdown: str, pdf_text_pdfplumber: str, pdf_text_docling: str):

    # 메시지 객체 생성
    system_msg = SystemMessage(_SYSTEM_PROMPT)
    human_prompt = get_prompt_pdf_text_to_markdown_validate(instruction_markdown, pdf_text_pdfplumber, pdf_text_docling)
    human_msg = HumanMessage(human_prompt)

    # 채팅 모델과 함께 사용
    messages = [system_msg, human_msg]

    llm = create_llm_model()
    response = llm.invoke(messages)  # AIMessage 반환

    return response


# pdf to markdown으로 정리된 지시서를 받아 확정분과 청구분으로 구분하여 데이터 정리
def validate_pdf_instruction_markdown_with_agent(instruction_markdown: str, pdf_text_pdfplumber: str, pdf_text_docling: str):

    tools = [write_todos, read_todos]
    llm = create_llm_model()
    human_prompt = get_prompt_pdf_text_to_markdown_validate(instruction_markdown, pdf_text_pdfplumber, pdf_text_docling)

    user_message = _SYSTEM_PROMPT + "\n\n" + "=" * 80 + "\n\n" + human_prompt

    agent = create_agent(
        llm,
        tools,
        system_prompt=_TODO_USAGE_INSTRUCTIONS,
        state_schema=DeepAgentState,
    )

    response = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": user_message
                }
            ],
        }
    )

    return response

# LLM - text to markdown 정리내용 검수

In [ ]:
# text to markdown으로 정리된 지시서를 받아 확정분과 청구분으로 구분하여 데이터 정리
def validate_text_instruction_markdown(instruction_markdown: str, original_text: str):

    # 메시지 객체 생성
    system_msg = SystemMessage(_SYSTEM_PROMPT)
    human_prompt = get_prompt_text_to_markdown_validate(instruction_markdown, original_text)
    human_msg = HumanMessage(human_prompt)

    # 채팅 모델과 함께 사용
    messages = [system_msg, human_msg]

    llm = create_llm_model()
    response = llm.invoke(messages)  # AIMessage 반환

    return response


# text to markdown으로 정리된 지시서를 받아 확정분과 청구분으로 구분하여 데이터 정리
def validate_text_instruction_markdown_with_agent(instruction_markdown: str, original_text: str):

    tools = [write_todos, read_todos]
    llm = create_llm_model()
    human_prompt = get_prompt_text_to_markdown_validate(instruction_markdown, original_text)

    user_message = _SYSTEM_PROMPT + "\n\n" + "=" * 80 + "\n\n" + human_prompt

    agent = create_agent(
        llm,
        tools,
        system_prompt=_TODO_USAGE_INSTRUCTIONS,
        state_schema=DeepAgentState,
    )

    response = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": user_message
                }
            ],
        }
    )

    return response

# LLM - 확정분과 청구분으로 구분하여 데이터 정리

In [ ]:
# markdown으로 정리된 지시서를 받아 확정분과 청구분으로 구분하여 데이터 정리
def classify_data_from_instruction(instruction_markdown: str):

    # 메시지 객체 생성
    system_msg = SystemMessage(_SYSTEM_PROMPT)
    human_prompt = get_prompt_confirmed_expected_clarification(instruction_markdown)
    human_msg = HumanMessage(human_prompt)

    # 채팅 모델과 함께 사용
    messages = [system_msg, human_msg]

    llm = create_llm_model()
    response = llm.invoke(messages)  # AIMessage 반환

    return response



### Agent

In [ ]:
def classify_data_from_instruction_with_agent_(instruction_markdown: str):

    # 메시지 객체 생성
    tools = [write_todos, read_todos]
    llm = create_llm_model()
    human_prompt = get_prompt_confirmed_expected_clarification(instruction_markdown)

    user_message = _SYSTEM_PROMPT + "\n\n" + "=" * 80 + "\n\n" + human_prompt

    agent = create_agent(
        llm,
        tools,
        system_prompt=_TODO_USAGE_INSTRUCTIONS,
        state_schema=DeepAgentState,
    )

    # response = agent.invoke(
    #     {
    #         "messages": [
    #             {
    #                 "role": "user",
    #                 "content": user_message
    #             }
    #         ],
    #         "todos": [],
    #     }
    # )

    response = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": user_message
                }
            ],
        }
    )

    return response

from langchain.agents.middleware import TodoListMiddleware

def classify_data_from_instruction_with_agent(instruction_markdown: str):

    
    # 메시지 객체 생성
    tools = [write_todos, read_todos]
    llm = create_llm_model()
    human_prompt = get_prompt_confirmed_expected_clarification(instruction_markdown)

    todo_agent = create_agent(
        llm,
        system_prompt=_SYSTEM_PROMPT,
        middleware=[TodoListMiddleware()]
    )

    # response = agent.invoke(
    #     {
    #         "messages": [
    #             {
    #                 "role": "user",
    #                 "content": user_message
    #             }
    #         ],
    #         "todos": [],
    #     }
    # )

    response = todo_agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": human_prompt
                }
            ],
        }
    )

    return response

# LLM - 확정분 데이터 수집

In [ ]:
def get_confirmed_data_from_instruction(instruction_markdown: str):

    # 메시지 객체 생성
    system_msg = SystemMessage(_SYSTEM_PROMPT)
    human_msg = HumanMessage(f"""
아래의 제공된 문서는 수익자가 보내온 변액일임펀드 설정/해지 지시 내용을 markdown 형식으로 작성한 문서입니다.

아래의 확정분과 청구분 구분 기준에 따라 확정분 데이터를 구분하세요. 

지침에 따라 제공된 문서에서 확정분 데이터를 수집하세요.
  
# 확정분과 청구분 구분 기준

{_CONFIRMED_EXPECTED_CLARIFICATION_PROMPT}

# 제공된 문서

{instruction_markdown}
  

# 반드시 지켜야 할 중요 지침

- 문서 전체 내용을 분석하세요.

- 문서에서 확정분과 청구분을 구분하는 기준에 따라 확정분과 청구분으로 설정/해지 데이터를 구분하세요.(모든 설정/해지 데이터는 반드시 확정분 또는 청구분 중 하나에 속해야 합니다.)

- 확정분을 구분하는 기준에 따라 확정분에 해당하는 모든 데이터를 수집하세요.

- 확정분으로 판단한 근거를 설명하세요.

- 확정분을 구분하는 기준에 따라 확정분에 해당하는 데이터가 없으면 데이터가 없음을 출력하세요.

- 확정분을 구분하는 기준에 따라 확정분에 해당하는 데이터가 없으면 더 이상 데이터를 수집하지 마세요.

- 확정분을 구분하는 기준에 따라 확정분에 해당하는 데이터가 없으면 데이터 없음과 판단한 근거를 설명하고 다른 내용은 출력하지 마세요.

- 통합 또는 요약하지 말고, 종목(펀드)명과 펀드코드 단위로 data를 정리하세요.

- 확정분 수집 결과에서 누락된 필드가 있는지 확인하세요. 누락된 필드가 있으면 추가하세요.

- 확정분 수집 결과에서 누락된 데이터가 있는지 확인하세요. 누락된 데이터가 있으면 추가하세요.

- 확정분 수집 결과에서 금액과 합계 데이터가 있는지 확인하세요. 합계가 정확한지 검증하세요.

- 확정분 수집 결과의 테이블에서 데이터가 인접한 컬럼에 중복되거나 병합되어 작성되어 있는지 확인하세요. 중복 또는 병합되어 있으면 수정하세요.

- 확정분 수집 결과에서 단어 또는 숫자를 임의로 제거하지 말고 있는 그대로 출력하세요.

- 확정분 수집 결과에서 단어 또는 숫자를 임의로 요약하지 말고 있는 그대로 출력하세요.

- 확정분 수집 결과에서 단어 또는 숫자를 임의로 새로 작성하지 말고 있는 그대로 출력하세요.

- 단어의 의미를 분석하고 맥락을 통해 공백, 띄어쓰기, 줄바꿈 오류가 존재하는지 확인하고 오류가 있으면 단어의 의미와 맥락에 맞도록 수정하세요.

- 종목명에서 공백, 띄어쓰기, 줄바꿈 오류가 자주 발생합니다. 종목명에서 공백, 띄어쓰기, 줄바꿈 오류가 존재하는지 확인하고 오류가 있으면 단어의 의미와 맥락에 맞도록 수정하세요.

- 종목명에서 약어를 사용하는지 판단하여 약어를 유지하세요.

- 종목명(펀드명)과 펀드코드가 정확한지 반드시 확인하세요.

- 확정분 수집 결과의 메타데이터(컬럼, 필드)를 빠짐없이 분석하여 의미와 기능을 설명하세요.

- 원문을 번역하지 말고 원문 그대로 출력하세요.


# 출력 데이터 검수(반드시 수행)

- 지침에 따라 정확하게 작성되었는지 확인한다.

- 확정분을 구분하는 기준에 따라 정확히 추출되었는지 확인한다.

- 확정분에 대한 필드와 텍스트가 정확히 추출되었는지 확인한다.

- 펀드 건수(N)와 출력 행 수가 일치하는지 확인한다.

- 모든 항목에서 종목명(펀드명)과 펀드코드가 정확하게 작성되었는지 재확인한다.

- 확정분을 구분하는 기준에 따라 확정분에 해당하는 데이터가 없으면 데이터가 없음으로 출력되었는지 확인한다.

- 확정분을 구분하는 기준에 따라 확정분에 해당하는 데이터가 없으면 더 이상 데이터를 수집하지 않았는지 확인한다.

- 확정분을 구분하는 기준에 따라 확정분에 해당하는 데이터가 없으면 데이터 없음과 근거만 출력되었는지 확인한다.

- Markdown 코드의 오류 여부를 검수하여 오류가 발견되면 수정한다.

- 오류가 있으면 수정한 뒤 검수 결과를 작성한다.

    """)

    # 채팅 모델과 함께 사용
    messages = [system_msg, human_msg]

    llm = create_llm_model()
    response = llm.invoke(messages)  # AIMessage 반환

    return response

# LLM - 청구분 수집

In [ ]:
def get_expected_data_from_instruction(instruction_markdown: str):

    # 메시지 객체 생성
    system_msg = SystemMessage(_SYSTEM_PROMPT)
    human_msg = HumanMessage(f"""
아래의 제공된 문서는 수익자가 보내온 변액일임펀드 설정/해지 지시 내용을 markdown 형식으로 작성한 문서입니다.

아래의 확정분과 청구분 구분 기준에 따라 청구분 데이터를 구분하세요. 

지침에 따라 제공된 문서에서 청구분 데이터를 수집하세요.
  
# 확정분과 청구분 구분 기준

{_CONFIRMED_EXPECTED_CLARIFICATION_PROMPT}


# 제공된 문서

{instruction_markdown}
  

# 반드시 지켜야 할 중요 지침

- 문서 전체 내용을 분석하세요.

- 문서에서 확정분과 청구분을 구분하는 기준에 따라 확정분과 청구분으로 데이터를 구분하세요.(모든 설정/해지 데이터는 반드시 확정분 또는 청구분 중 하나에 속해야 합니다.)

- 청구분을 구분하는 기준에 따라 청구분에 해당하는 모든 데이터를 수집하세요.

- 청구분으로 판단한 근거를 설명하세요.

- 청구분을 구분하는 기준에 따라 청구분에 해당하는 데이터가 없으면 데이터가 없음을 출력하세요.

- 청구분을 구분하는 기준에 따라 청구분에 해당하는 데이터가 없으면 더 이상 데이터를 수집하지 마세요.

- 청구분을 구분하는 기준에 따라 청구분에 해당하는 데이터가 없으면 데이터 없음과 판단한 근거를 설명하고 다른 내용은 출력하지 마세요.

- 통합 또는 요약하지 말고, 종목(펀드)명과 펀드코드 단위로 data를 정리하세요.

- 청구분 수집 결과에서 누락된 필드가 있는지 확인하세요. 누락된 필드가 있으면 추가하세요.

- 청구분 수집 결과에서 누락된 데이터가 있는지 확인하세요. 누락된 데이터가 있으면 추가하세요.

- 청구분 수집 결과에서 금액과 합계 데이터가 있는지 확인하세요. 합계가 정확한지 검증하세요.

- 청구분 수집 결과에서 단어 또는 숫자를 임의로 제거하지 말고 있는 그대로 출력하세요.

- 청구분 수집 결과에서 단어 또는 숫자를 임의로 요약하지 말고 있는 그대로 출력하세요.

- 청구분 수집 결과에서 단어 또는 숫자를 임의로 새로 작성하지 말고 있는 그대로 출력하세요.

- 단어의 의미를 분석하고 맥락을 통해 공백, 띄어쓰기, 줄바꿈 오류가 존재하는지 확인하고 오류가 있으면 단어의 의미와 맥락에 맞도록 수정하세요.

- 종목명에서 공백, 띄어쓰기, 줄바꿈 오류가 자주 발생합니다. 종목명에서 공백, 띄어쓰기, 줄바꿈 오류가 존재하는지 확인하고 오류가 있으면 단어의 의미와 맥락에 맞도록 수정하세요.

- 종목명에서 약어를 사용하는지 판단하여 약어를 유지하세요.

- 종목명(펀드명)과 펀드코드가 정확한지 반드시 확인하세요.

- 청구분 수집 결과의 메타데이터(컬럼, 필드)를 빠짐없이 분석하여 의미와 기능을 설명하세요.

- 원문을 번역하지 말고 원문 그대로 출력하세요.


# 출력 데이터 검수(반드시 수행)

- 지침에 따라 정확하게 작성되었는지 확인한다.

- 청구분을 구분하는 기준에 따라 정확히 추출되었는지 확인한다.

- 청구분에 대한 필드와 텍스트가 정확히 추출되었는지 확인한다.

- 펀드 건수(N)와 출력 행 수가 일치하는지 확인한다.

- 모든 항목에서 종목명(펀드명)과 펀드코드가 정확하게 작성되었는지 재확인한다.

- 청구분을 구분하는 기준에 따라 청구분에 해당하는 데이터가 없으면 데이터가 없음으로 출력되었는지 확인한다.

- 청구분을 구분하는 기준에 따라 청구분에 해당하는 데이터가 없으면 더 이상 데이터를 수집하지 않았는지 확인한다.

- 청구분을 구분하는 기준에 따라 청구분에 해당하는 데이터가 없으면 데이터 없음과 근거만 출력되었는지 확인한다.

- Markdown 코드의 오류 여부를 검수하여 오류가 발견되면 수정한다.

- 오류가 있으면 수정한 뒤 검수 결과를 작성한다.

    """)

    # 채팅 모델과 함께 사용
    messages = [system_msg, human_msg]

    llm = create_llm_model()
    response = llm.invoke(messages)  # AIMessage 반환

    return response

# Graph State

In [ ]:
# Graph state 클래스

from typing_extensions import TypedDict

class State(TypedDict):
    file_path: str # 변액일임펀드 설정/해지 지시서 파일 경로
    file_name: str # 지시서 파일 이름
    file_ext: str # 지시서 파일 확장자
    password: str # 지시서 암호
    original_text_docling: str # 지시서 원본 텍스트 - docling
    original_text_pdfplumber: str # 지시서 원본 텍스트 - pdfplumber
    original_text: str # 지시서 원본 텍스트
    markdown_text: str # 지시서 마크다운 텍스트
    relevance_score: str # 지시서 여부 점수 : yes, no
    confirmed_expected_data: str # 지시서 확정분과 청구분 구분 결과
    expected_data: str # 지시서 청구분 데이터
    confirmed_data: str # 지시서 확정분 데이터
    expected_subscription_data: str # 지시서 청구분 설정 데이터
    expected_redemption_data: str # 지시서 청구분 해지 데이터
    confirmed_subscription_data: str # 지시서 확정분 설정 데이터
    confirmed_redemption_data: str # 지시서 확정분 해지 데이터

# Node - 문서 정보 수집

In [ ]:
from pathlib import Path

def node_get_document_info(state: State) -> State:
    file_path = state["file_path"]
    
    # file_path에서 파일명과 확장자 추출    
    path_obj = Path(file_path)
    file_name = path_obj.name  # 파일명 (확장자 포함)
    file_ext = path_obj.suffix  # 확장자 (점 포함, 예: .pdf)

    print(f"   file_path : {file_path}")
    print(f"   file_name : {file_name}")
    print(f"   file_ext : {file_ext}")
    
    return {"file_name": file_name, "file_ext": file_ext}


In [ ]:
# 노드 - 문서(pdf, xls, xlsx) 로드
def node_load_document(state: State) -> State:
    file_path = state["file_path"]
    password = state["password"]
    original_text = load_document(file_path, password)
    return {"original_text": original_text}

# Node - PDF 문서 파싱 - docling

In [ ]:
# 노드 - PDF 문서 파싱 - docling
def node_parse_pdf_document_docling(state: State) -> State:
    file_path = state["file_path"]
    password = state["password"]
    pdf_text = extract_pdf_with_docling(file_path, password)

    print("docling")
    # print(pdf_text)

    return {"original_text_docling": pdf_text}


# Node - PDF 문서 파싱 - pdfplumber

In [ ]:
# 노드 - PDF 문서 파싱 - pdfplumber
def node_parse_pdf_document_pdfplumber(state: State) -> State:
    file_path = state["file_path"]
    password = state["password"]
    pdf_text = extract_pdf_with_pdfplumber(file_path, password)

    print("pdfplumber")
    # print(pdf_text)

    return {"original_text_pdfplumber": pdf_text}

# Node - PDF 추출 텍스트 병합

In [ ]:
def node_merge_pdf_text(state: State) -> State:
    pdfplumber_text = state["original_text_pdfplumber"]
    docling_text = state["original_text_docling"]
    if docling_text == "":
        state["original_text_docling"] = pdfplumber_text
        print("docling_text is empty")

    print("merge")
    # print(pdfplumber_text)

    return {"original_text": pdfplumber_text}

# Node - excel 문서 파서

In [ ]:
def node_parse_excel_document(state: State) -> State:
    file_path = state["file_path"]
    original_text = parser_excel(file_path)

    # print("****************excel parser****************")
    # print(f"   {original_text}")
    
    return {"original_text": original_text}


# Node - 문서 관련성 체크

In [ ]:
# 노드 - 문서 관련성 체크 노드
def node_check_relevance(state: State) -> State:
    original_text = state["original_text"]
    # chain_retrieval_grader = create_chain_retrieval_grader()
    # relevance_score = chain_retrieval_grader.invoke({"original_text": original_text})
    relevance_score = "yes"
    # print("****************check_relevance****************")
    # print(f"   {relevance_score}")
    return {"relevance_score": relevance_score}
    # return {"relevance_score": relevance_score.binary_score}


# Node - markdown 변환

In [ ]:
def node_convert_text_to_markdown(state: State) -> State:
    print("****************START node_convert_text_to_markdown****************")
    file_ext = state["file_ext"]
    if file_ext == ".pdf":
        document_text_docling = state["original_text_docling"]
        document_text_pdfplumber = state["original_text_pdfplumber"]
        # response = convert_pdf_text_to_markdown_using_agent(document_text_docling, document_text_pdfplumber)
        response = convert_pdf_text_to_markdown(document_text_docling, document_text_pdfplumber)        
        print("****************pdf markdown****************")
        # markdown_text = response.content
        # print(f"   {markdown_text}")
    elif file_ext == ".xlsx" or state["file_ext"] == ".xls":
        original_text = state["original_text"]
        # response = convert_excel_text_to_markdown_using_agent(original_text)        
        response = convert_excel_text_to_markdown(original_text)
        print("****************excel markdown****************")
        # markdown_text = response.content
        # print(f"   {markdown_text}")
    else:
        original_text = state["original_text"]
        # response = convert_excel_text_to_markdown_using_agent(original_text)
        response = convert_excel_text_to_markdown(original_text)        
        print("****************unknown markdown****************")
        # markdown_text = response.content
        # print(f"   {markdown_text}")

    # last_ai_message = get_last_ai_message(response)
    # markdown_text = last_ai_message.content

    markdown_text = response.content

    return {"markdown_text": markdown_text}


# Node - markdown 검수

In [ ]:
def node_validate_markdown(state: State) -> State:
    print("****************START node_validate_markdown****************")
    markdown_text = state["markdown_text"]
    file_ext = state["file_ext"]
    if file_ext == ".pdf":
        pdf_text_pdfplumber = state["original_text_pdfplumber"]
        pdf_text_docling = state["original_text_docling"]
        validate_markdown = validate_pdf_instruction_markdown_with_agent(markdown_text, pdf_text_pdfplumber, pdf_text_docling)
        # validate_markdown = validate_pdf_instruction_markdown(markdown_text, pdf_text_pdfplumber, pdf_text_docling)
    elif file_ext == ".xlsx" or file_ext == ".xls":
        original_text = state["original_text"]
        validate_markdown = validate_text_instruction_markdown_with_agent(markdown_text, original_text)
        # validate_markdown = validate_text_instruction_markdown(markdown_text, original_text)
    else:
        original_text = state["original_text"]
        validate_markdown = validate_text_instruction_markdown_with_agent(markdown_text, original_text)
        # validate_markdown = validate_text_instruction_markdown(markdown_text, original_text)

    last_ai_message = get_last_ai_message(validate_markdown)
    validate_markdown_text = last_ai_message.content
    # validate_markdown_text = validate_markdown.content

    return {"markdown_text": validate_markdown_text}


# Node - 확정분과 청구분 구분 정리

In [ ]:
def node_classify_confirmed_expected_data(state: State) -> State:
    print("****************START node_classify_confirmed_expected_data****************")

    markdown_text = state["markdown_text"]
    # confirmed_expected_data = classify_data_from_instruction(markdown_text)
    confirmed_expected_data = classify_data_from_instruction_with_agent(markdown_text)
    last_ai_message = get_last_ai_message(confirmed_expected_data)
    # print(f"   {confirmed_expected_data}")
    return {"confirmed_expected_data": last_ai_message.content}

# Node - 청구분 수집

In [ ]:
def node_get_confirmed_data(state: State) -> State:
    print("****************START node_get_confirmed_data****************")

    markdown_text = state["markdown_text"]
    confirmed_data = get_confirmed_data_from_instruction(markdown_text)
        
    return {"confirmed_data": confirmed_data.content}

# Node - 확정분 수집

In [ ]:
def node_get_expected_data(state: State) -> State:
    print("****************START node_get_expected_data****************")

    markdown_text = state["markdown_text"]
    expected_data = get_expected_data_from_instruction(markdown_text)
        
    return {"expected_data": expected_data.content}

# Node - 청구분 설정건 수집

# Node - 청구분 해지건 수집

# Node - 확정분 설정건 수집

# Node - 확정분 해지건 수집

# Node - 수집 완료 데이터 병합

In [ ]:
def node_merge_collected_data(state: State) -> State:
    print("****************START node_merge_collected_data****************")
    return


# Node - 완료 처리

In [ ]:
def node_complete(state: State) -> State:
    print("****************END****************")
    return


# Node - Fail 처리

In [ ]:
def node_no_relevance(state: State) -> State:
    print("**************** node_no_relevance ****************")    
    return {"markdown_text": ""}


# Graph 구성

### graph 생성

In [ ]:
# Graph 생성

from langgraph.graph import StateGraph, START, END

graph_builder = StateGraph(State)

### graph node 생성

In [ ]:
# 지시서 파일 저 노드 생성
graph_builder.add_node("get_document_info", node_get_document_info)

# pdf 문서 파싱 노드 생성 - docling
graph_builder.add_node("parse_pdf_document_docling", node_parse_pdf_document_docling)

# pdf 문서 파싱 노드 생성 - pdfplumber
graph_builder.add_node("parse_pdf_document_pdfplumber", node_parse_pdf_document_pdfplumber)

# pdf 문서 파싱 정보 취합 노드 생성
graph_builder.add_node("merge_pdf_text", node_merge_pdf_text)

# 엑셀 문서 파싱 노드 생성
graph_builder.add_node("parse_excel_document", node_parse_excel_document)

# 문서 관련성 체크 노드 생성
graph_builder.add_node("check_relevance", node_check_relevance)

# 문서 텍스트 마크다운 변환 노드 생성
graph_builder.add_node("convert_text_to_markdown", node_convert_text_to_markdown)

# markdown 검수 노드 생성
graph_builder.add_node("validate_markdown", node_validate_markdown)

# 확정분과 청구분 구분 정리 노드 생성
graph_builder.add_node("classify_confirmed_expected_data", node_classify_confirmed_expected_data)

# 확정분 수집 노드 생성
graph_builder.add_node("get_confirmed_data", node_get_confirmed_data)

# 청구분 수집 노드 생성
graph_builder.add_node("get_expected_data", node_get_expected_data)

# 수집 완료 데이터 병합 노드 생성
graph_builder.add_node("merge_collected_data", node_merge_collected_data)

# 문서 관련성 없음 노드 생성
graph_builder.add_node("no_relevance", node_no_relevance)

# 완료 노드 생성
graph_builder.add_node("complete", node_complete)

### graph edge 설정

In [ ]:
# graph 엣지 설정

graph_builder.add_edge(START, "get_document_info")

# 파일 확장자에 따른 노드 라우팅
def routing_parser(state: State):
    print("****************routing_parser****************")
    print(f"   {state['file_ext']}")
    file_ext = state["file_ext"]
    if file_ext == ".pdf":
        # 문서 파싱 노드 라우팅 - 병렬 실행
        return ["parse_pdf_document_docling", "parse_pdf_document_pdfplumber"]
    elif file_ext == ".xlsx" or file_ext == ".xls":
        # 엑셀 문서 파싱 노드 라우팅
        return "parse_excel_document"
    else:
        return "complete"

graph_builder.add_conditional_edges("get_document_info", routing_parser)

graph_builder.add_edge("parse_pdf_document_docling", "merge_pdf_text")
graph_builder.add_edge("parse_pdf_document_pdfplumber", "merge_pdf_text")

graph_builder.add_edge("merge_pdf_text", "check_relevance")
graph_builder.add_edge("parse_excel_document", "check_relevance")

def routing_check_relevance(state: State):
    relevance_score = state["relevance_score"]
    print("****************routing_check_relevance****************")
    print(f"   relevance_score : {relevance_score}")
    if relevance_score == "yes":
        print("****************routing_check_relevance yes****************")
        print(f"   routing to  convert_text_to_markdown")
        return "convert_text_to_markdown"
        # return "no_relevance"
    else:
        print("****************routing_check_relevance no****************")
        print(f"   routing to fail")
        return "no_relevance"

graph_builder.add_conditional_edges("check_relevance", routing_check_relevance)

graph_builder.add_edge("convert_text_to_markdown", "validate_markdown")
# graph_builder.add_edge("validate_markdown", "merge_collected_data")
graph_builder.add_edge("validate_markdown", "classify_confirmed_expected_data")
# graph_builder.add_edge("convert_text_to_markdown", "classify_confirmed_expected_data")
graph_builder.add_edge("classify_confirmed_expected_data", "merge_collected_data")

def routing_parallel_confirmed_expected(state: State):
    print("****************routing_parallel_confirmed_expected****************")
    return ["get_confirmed_data", "get_expected_data"]

# graph_builder.add_conditional_edges("convert_text_to_markdown", routing_parallel_confirmed_expected)
# graph_builder.add_edge("get_confirmed_data", "merge_collected_data")
# graph_builder.add_edge("get_expected_data", "merge_collected_data")


graph_builder.add_edge("merge_collected_data", "complete")

graph_builder.add_edge("no_relevance", "complete")

graph_builder.add_edge("complete", END)



### graph 컴파일

In [ ]:
# graph 컴파일

graph = graph_builder.compile()

In [ ]:
from langchain_teddynote.graphs import visualize_graph

# 그래프 시각화
visualize_graph(graph, xray=True)

# Graph 실행

In [ ]:
# graph 실행

# text 추출 태스트

_document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/라이나_250826.xlsx"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프(2차)_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프(액티브)_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프(퇴직)_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/카디프_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/하나생명(액티브)_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/iM라이프_250826.xls"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/ABL_250826.xlsx"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/DB_250826.xlsx"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/KB라이프_250826.xls"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/KB라이프(액티브)_250826.xls"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_overseas_settlement/LS.pdf"

_password = None
# _password = '345678'
result = graph.invoke({"file_path": _document_file_path, "password": _password})

# print(result)
result_markdown_text = result["markdown_text"]



# Markdown 정리 결과

In [ ]:
from IPython.display import Markdown, display

print(f"file_name : {result['file_name']}")
# print(result_markdown_text)
display(Markdown(result_markdown_text))

# 확정분과 청구분 구분 정리

In [ ]:
display(Markdown(result["confirmed_expected_data"]))

In [ ]:
# display(Markdown(result["markdown_text"]))

# 확정분 수집 결과

In [ ]:
# display(Markdown(result["confirmed_data"]))

# 청구분 수집 결과

In [ ]:
# display(Markdown(result["expected_data"]))